# Quantum-Enhanced Fog Learning for Early Pulmonary Disorder Detection
### A hybrid quantum–classical, fog–cloud clinical decision-support system with explainable AI

**Abstract.** Chest X-rays are the most common imaging test, and most of them are normal. Sending every image to a powerful cloud model is slow
and costly, and deep models rarely explain their output. This project builds a two-tier system:
a lightweight **fog node** (MobileNetV3-Small, CPU) screens every X-ray and keeps low-risk cases locally (a screening result, never a clearance); only flagged cases travel over the
network to the **cloud tier**, where a DenseNet121 backbone, an **8-qubit PennyLane variational quantum circuit** and **cross-attention fusion**
predict 14 thoracic disorders. Every cloud diagnosis comes with **Grad-CAM++ heatmaps computed through the quantum circuit**, a
quantum-attention map, and a **plain-language reasoning report** (anatomical-zone analysis, plausibility check, occlusion test, shortcut warning).
All models are evaluated on a **patient-level** held-out test set with a matched **classical control**, multiple seeds and bootstrap confidence intervals.

| Part | Cells | Content |
|---|---|---|
| A | 1–4 | Setup, dataset download, imports, configuration |
| B | 5–9 | Dataset audit and exploratory data analysis |
| C | 10–12 | Preprocessing, patient-level split, augmentation |
| D | 13–15 | Phase 1 — cloud backbone (DenseNet121) baseline |
| E | 16–19 | Phase 2 — quantum layer and the **16 → 8 qubit** study |
| F | 20–23 | Hybrid models vs classical control (3 seeds, bootstrap CIs) |
| G | 24–27 | Phase 3 — fog-tier triage (MobileNetV3-Small) |
| H | 28–32 | Fog ↔ cloud integration: live HTTP services, network emulation, fallback |
| I | 33–37 | Explainable AI with reasoning |
| J | 38–41 | End-to-end clinical reports, results summary, saved artefacts |
| K | 42 | **Live demo dashboard** (public link): upload an X-ray and watch the system work |
| L | — | Conclusion, limitations, viva questions |

> **Academic research prototype — not a medical device.** Outputs are model estimates on a public benchmark and must not be used for patient care.

## Part A · Setup

In [ ]:
# ===== Cell 1: Environment, Drive and packages =====
import os, sys, subprocess, glob, copy, time, json, shutil, io, base64, threading, warnings
warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/qfog_project'
    DATA_ROOT = '/content/nih_sample'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pennylane', 'grad-cam', 'kaggle', 'gradio',
                    'fastapi', 'uvicorn', 'python-multipart', 'seaborn'], check=True)
else:
    PROJECT_DIR = os.environ.get('QFOG_DIR', os.path.abspath('qfog_run'))
    DATA_ROOT = os.environ['QFOG_DATA']
SMOKE = os.environ.get('QFOG_SMOKE') == '1'          # tiny settings, used only for offline testing
OUT = f'{PROJECT_DIR}/qfog_complete'
for d in ('checkpoints', 'results', 'figures', 'cache', 'demo_examples'):
    os.makedirs(f'{OUT}/{d}', exist_ok=True)
FIG = f'{OUT}/figures'
print('Outputs will be saved in', OUT)

In [ ]:
# ===== Cell 2: Download the NIH ChestX-ray14 sample (Kaggle) =====
# First run asks for kaggle.json (kaggle.com -> Settings -> API -> Create New Token) and keeps a copy on Drive.
if not glob.glob(f'{DATA_ROOT}/**/*.png', recursive=True):
    assert IN_COLAB, f'No images under {DATA_ROOT}'
    key = f'{PROJECT_DIR}/kaggle.json'
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    if not os.path.exists(key):
        from google.colab import files
        print('Upload kaggle.json'); up = files.upload(); shutil.copy(next(iter(up)), key)
    shutil.copy(key, os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    os.makedirs(DATA_ROOT, exist_ok=True)
    print('Downloading NIH sample (about 3-6 minutes)...')
    subprocess.run(f'kaggle datasets download -d nih-chest-xrays/sample -p {DATA_ROOT} --unzip', shell=True, check=True)
print(len(glob.glob(f'{DATA_ROOT}/**/*.png', recursive=True)), 'X-ray images on disk')

In [ ]:
# ===== Cell 3: Imports =====
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models, torchvision.transforms.v2 as T
import pennylane as qml
import matplotlib
if not IN_COLAB: matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| pennylane', qml.__version__, '| device:', device)

In [ ]:
# ===== Cell 4: Configuration =====
CLASSES = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
           'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
CFG = dict(
    backbone='densenet121',          # CheXNet-style backbone for the cloud ('resnet18' also supported)
    pretrained=not SMOKE,
    baseline_epochs=1 if SMOKE else 15,
    triage_epochs=1 if SMOKE else 12,
    head_epochs=2 if SMOKE else 30,
    seeds=[42] if SMOKE else [42, 7, 123],
    workers=0 if SMOKE else 2,
    n_qubits=8, n_layers=3,
    triage_target_recall=0.85,       # fog threshold is chosen on VALIDATION to reach this recall
    min_test_positives=15,           # classes with fewer test positives are flagged as statistically unreliable
)
REUSE_SAVED = True                   # True: reuse models saved by an earlier run (fast demo). False: retrain everything.
AMP = device.type == 'cuda'

def set_seed(s):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
CFG

## Part B · Dataset audit and exploratory data analysis
**Dataset:** NIH ChestX-ray14 (Wang et al., CVPR 2017), Kaggle *sample* subset — 5,606 frontal chest X-rays from the NIH Clinical Center with
14 disease labels mined from radiology reports (an image can have several labels, or "No Finding"). Labels are NLP-derived, so some label noise is expected.

In [ ]:
# ===== Cell 5: Load labels and audit the dataset =====
csv_path = sorted(c for c in glob.glob(f'{DATA_ROOT}/**/*.csv', recursive=True) if 'label' in os.path.basename(c).lower())[0]
img_path = {}
for p in glob.glob(f'{DATA_ROOT}/**/*.png', recursive=True):
    img_path.setdefault(os.path.basename(p), p)
df = pd.read_csv(csv_path)
for c in CLASSES:
    df[c] = df['Finding Labels'].apply(lambda s, c=c: int(c in s.split('|')))
df['abnormal'] = (df[CLASSES].sum(axis=1) > 0).astype(int)
df['n_labels'] = df[CLASSES].sum(axis=1)
if 'Patient Age' in df:
    df['age'] = pd.to_numeric(df['Patient Age'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
missing = [n for n in df['Image Index'] if n not in img_path]
audit = pd.Series({'images (rows)': len(df), 'unique patients': df['Patient ID'].nunique(),
                   'images found on disk': len(df) - len(missing), 'abnormal (>= 1 finding)': int(df['abnormal'].sum()),
                   'normal (No Finding)': int((df['abnormal'] == 0).sum()), 'images with 2+ findings': int((df['n_labels'] >= 2).sum()),
                   'duplicate image names': int(df['Image Index'].duplicated().sum())})
assert not missing, f'{len(missing)} images missing'
print('Columns:', list(df.columns[:12]))
display_df = df.head(8)
audit.to_frame('value')

In [ ]:
# ===== Cell 6: EDA: class distribution, normal vs abnormal, labels per image =====
counts = df[CLASSES].sum().sort_values(ascending=False)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2), gridspec_kw={'width_ratios': [2.2, 1, 1]})
sns.barplot(x=counts.values, y=counts.index, ax=ax[0], color='steelblue'); ax[0].set_title('Images per finding (multi-label)')
for i, v in enumerate(counts.values): ax[0].text(v, i, f' {v}', va='center', fontsize=8)
df['abnormal'].map({0: 'Normal', 1: 'Abnormal'}).value_counts().plot.pie(ax=ax[1], autopct='%1.0f%%', colors=['#8fbc8f', '#cd5c5c'])
ax[1].set_ylabel(''); ax[1].set_title('Normal vs abnormal')
df['n_labels'].value_counts().sort_index().plot.bar(ax=ax[2], color='gray'); ax[2].set_title('Findings per image'); ax[2].set_xlabel('number of findings')
plt.tight_layout(); plt.savefig(f'{FIG}/eda_class_distribution.png', dpi=150); plt.show()
print('Class imbalance: most common', counts.index[0], counts.iloc[0], '| rarest', counts.index[-1], counts.iloc[-1])

In [ ]:
# ===== Cell 7: EDA: co-occurrence of findings and patient demographics =====
co_arr = df[CLASSES].values.T @ df[CLASSES].values; np.fill_diagonal(co_arr, 0); co = pd.DataFrame(co_arr, index=CLASSES, columns=CLASSES)
ncols = 2 if 'age' in df else 1
fig, ax = plt.subplots(1, ncols + (1 if 'Patient Gender' in df else 0), figsize=(18 if ncols == 2 else 8, 5), squeeze=False)
sns.heatmap(co, ax=ax[0, 0], cmap='mako', annot=False); ax[0, 0].set_title('How often two findings appear together')
k = 1
if 'age' in df:
    sns.histplot(data=df, x='age', hue=df['abnormal'].map({0: 'Normal', 1: 'Abnormal'}), bins=30, element='step', ax=ax[0, k])
    ax[0, k].set_title('Patient age'); k += 1
if 'Patient Gender' in df:
    pd.crosstab(df['Patient Gender'], df['abnormal'].map({0: 'Normal', 1: 'Abnormal'})).plot.bar(ax=ax[0, k]); ax[0, k].set_title('Gender')
plt.tight_layout(); plt.savefig(f'{FIG}/eda_cooccurrence_demographics.png', dpi=150); plt.show()
if 'View Position' in df: print('View positions:', df['View Position'].value_counts().to_dict())

In [ ]:
# ===== Cell 8: EDA: example X-rays for each finding =====
fig, ax = plt.subplots(3, 5, figsize=(16, 10))
for a, c in zip(ax.ravel(), ['No Finding'] + CLASSES):
    rows = df[df['abnormal'] == 0] if c == 'No Finding' else df[df[c] == 1]
    if len(rows):
        a.imshow(Image.open(img_path[rows.iloc[0]['Image Index']]).convert('L'), cmap='gray')
    a.set_title(c, fontsize=10); a.axis('off')
plt.suptitle('One example per class (NIH ChestX-ray14 sample)'); plt.tight_layout()
plt.savefig(f'{FIG}/eda_examples.png', dpi=120); plt.show()

In [ ]:
# ===== Cell 9: EDA: image properties (size, intensity) =====
sample_names = df['Image Index'].sample(min(200, len(df)), random_state=0)
props = []
for n in sample_names:
    im = Image.open(img_path[n]).convert('L'); a = np.asarray(im)
    props.append({'width': im.size[0], 'height': im.size[1], 'mean intensity': a.mean(), 'std intensity': a.std()})
props = pd.DataFrame(props)
print('Original resolution:', props[['width', 'height']].mode().iloc[0].to_dict(), '-> resized to 256 (cache) and 224 (model input)')
props.describe().round(1)

## Part C · Preprocessing and patient-level split
* **Patient-level split (70/10/20):** the same patient never appears in two sets — otherwise the model can memorise a patient and look better than it is.
* **Validation** is used only to pick checkpoints and thresholds; **test** is used once, for the reported numbers.
* Images are decoded once to 256×256 grayscale and cached; the model sees 224×224, ImageNet-normalised, 3-channel inputs.
* Training augmentation is mild and anatomy-preserving (small crop, ±7° rotation, brightness/contrast). No horizontal flip: it would swap the heart side.

In [ ]:
# ===== Cell 10: Patient-level train / validation / test split =====
g1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trval_idx, test_idx = next(g1.split(df, groups=df['Patient ID']))
trval = df.iloc[trval_idx]
g2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=42)
tr_idx, va_idx = next(g2.split(trval, groups=trval['Patient ID']))
splits = {'train': trval.iloc[tr_idx].reset_index(drop=True), 'val': trval.iloc[va_idx].reset_index(drop=True),
          'test': df.iloc[test_idx].reset_index(drop=True)}
pats = {k: set(v['Patient ID']) for k, v in splits.items()}
assert not (pats['train'] & pats['val'] or pats['train'] & pats['test'] or pats['val'] & pats['test']), 'patient leakage!'
split_table = pd.DataFrame({k: {'images': len(v), 'patients': v['Patient ID'].nunique(), 'abnormal %': round(100 * v['abnormal'].mean(), 1)}
                            for k, v in splits.items()}).T
for k, v in splits.items(): v.to_csv(f'{OUT}/results/split_{k}.csv', index=False)
print('No patient appears in more than one split.'); split_table

In [ ]:
# ===== Cell 11: Image cache, datasets and augmentation =====
def load_gray(src):
    """Path / PIL image -> 256x256 grayscale uint8 (identical preprocessing for training, fog node and cloud)."""
    img = (src if isinstance(src, Image.Image) else Image.open(src)).convert('L').resize((256, 256), Image.BILINEAR)
    return np.array(img, dtype=np.uint8)

def build_cache(d, name):
    path = f'{OUT}/cache/{name}_256.npy'
    if os.path.exists(path):
        a = np.load(path)
        if len(a) == len(d): return a
    a = np.stack([load_gray(img_path[n]) for n in d['Image Index']]); np.save(path, a); return a

imgs = {k: build_cache(v, k) for k, v in splits.items()}
Y = {k: splits[k][CLASSES].values.astype(np.float32) for k in splits}
YA = {k: splits[k]['abnormal'].values.astype(np.float32) for k in splits}
reliable = [c for i, c in enumerate(CLASSES) if Y['test'][:, i].sum() >= CFG['min_test_positives']]
reliable = reliable or [c for i, c in enumerate(CLASSES) if Y['test'][:, i].sum() > 0]
print('Reliable classes (>= %d positives in test):' % CFG['min_test_positives'], reliable)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
AUG = T.Compose([T.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.95, 1.05), antialias=True),
                 T.RandomRotation(7), T.ColorJitter(brightness=0.1, contrast=0.1)])
def to_input(gray):
    t = torch.from_numpy(np.array(gray)).float().div(255)[None, None]
    t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)[0]
    return (t.expand(3, -1, -1) - MEAN) / STD
def denormalize(t):
    return (t.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

class CXRDataset(Dataset):
    def __init__(self, images, labels, abnormal, train=False):
        self.images, self.labels, self.abn, self.train = images, torch.tensor(labels), torch.tensor(abnormal), train
    def __len__(self): return len(self.images)
    def __getitem__(self, i):
        if self.train:
            t = AUG(torch.from_numpy(np.array(self.images[i])).float().div(255)[None]); t = (t.expand(3, -1, -1) - MEAN) / STD
        else:
            t = to_input(self.images[i])
        return t, self.labels[i], self.abn[i]
ds = {k: CXRDataset(imgs[k], Y[k], YA[k], train=(k == 'train')) for k in splits}
ds_eval = {k: CXRDataset(imgs[k], Y[k], YA[k]) for k in splits}
loader = lambda d, shuffle=False, bs=32: DataLoader(d, bs, shuffle=shuffle, num_workers=CFG['workers'])

fig, ax = plt.subplots(1, 5, figsize=(15, 3.2))
ax[0].imshow(imgs['train'][0], cmap='gray'); ax[0].set_title('original (256 px)')
for j in range(1, 5): ax[j].imshow(denormalize(ds['train'][0][0])); ax[j].set_title(f'augmented #{j}')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(f'{FIG}/augmentation.png', dpi=120); plt.show()

In [ ]:
# ===== Cell 12: Training and evaluation utilities =====
def fit(model, train_loader, val_loader, step, epochs, lr, patience=3, name='model', amp=False):
    """AdamW + early stopping. Keeps a deep copy of the best-validation weights (not a reference)."""
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    best, best_state, best_epoch, bad, hist = float('inf'), None, -1, 0, {'train': [], 'val': []}
    for ep in range(epochs):
        t0 = time.time(); model.train(); tl, n = 0.0, 0
        for b in train_loader:
            with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                loss, bs = step(b)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * bs; n += bs
        model.eval(); vl, m = 0.0, 0
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=amp):
            for b in val_loader:
                loss, bs = step(b); vl += loss.item() * bs; m += bs
        sched.step(); tl, vl = tl / n, vl / m
        hist['train'].append(tl); hist['val'].append(vl)
        print(f'  [{name}] epoch {ep:2d}  train {tl:.4f}  val {vl:.4f}  ({time.time() - t0:.0f}s)')
        if vl < best - 1e-4:
            best, best_state, best_epoch, bad = vl, copy.deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
            if bad >= patience: print('  early stopping'); break
    model.load_state_dict(best_state)
    print(f'  [{name}] best epoch {best_epoch}  (val loss {best:.4f})')
    return hist, best_epoch

def per_class_auroc(y, p):
    return {c: roc_auc_score(y[:, i], p[:, i]) for i, c in enumerate(CLASSES) if 0 < y[:, i].sum() < len(y)}
def mean_auroc(y, p, classes=None):
    a = per_class_auroc(y, p); classes = classes or list(a)
    return float(np.mean([a[c] for c in classes if c in a]))
def paired_bootstrap(y, pa, pb, classes, n=1000, seed=0):
    """Difference in mean AUROC (b - a) on resampled test images, with a 95% confidence interval."""
    rng = np.random.default_rng(seed); idx = [CLASSES.index(c) for c in classes]; d = []
    for _ in range(100 if SMOKE else n):
        s = rng.integers(0, len(y), len(y)); A, B = [], []
        for i in idx:
            if 0 < y[s, i].sum() < len(s):
                A.append(roc_auc_score(y[s, i], pa[s, i])); B.append(roc_auc_score(y[s, i], pb[s, i]))
        if A: d.append(np.mean(B) - np.mean(A))
    lo, hi = (np.percentile(d, [2.5, 97.5]) if d else (np.nan, np.nan))
    return mean_auroc(y, pb, classes) - mean_auroc(y, pa, classes), (float(lo), float(hi))
def n_params(m): return sum(p.numel() for p in m.parameters())

## Part D · Phase 1 — Cloud backbone baseline (DenseNet121)
DenseNet121 is the architecture used by CheXNet (Rajpurkar et al., 2017) for ChestX-ray14. It is pre-trained on ImageNet and fine-tuned for
14-label classification (binary cross-entropy per label). Its last feature map (7×7×1024) provides **49 spatial tokens** used later by the
attention fusion and by Grad-CAM++.

In [ ]:
# ===== Cell 13: Train the cloud backbone =====
class CloudBackbone(nn.Module):
    def __init__(self, arch='densenet121', pretrained=True):
        super().__init__()
        if arch == 'densenet121':
            d = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT if pretrained else None)
            self.features, self.dim = d.features, 1024
        else:
            r = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
            self.features, self.dim = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3, r.layer4), 512
        self.fc = nn.Linear(self.dim, len(CLASSES))
    @property
    def cam_layer(self): return self.features[-1]
    def tokens(self, x):
        f = F.relu(self.features(x)); return f.flatten(2).transpose(1, 2), f.mean(dim=(2, 3))
    def forward(self, x): return self.fc(self.tokens(x)[1])

set_seed(42)
backbone = CloudBackbone(CFG['backbone'], CFG['pretrained']).to(device)
crit = nn.BCEWithLogitsLoss()
def step_bb(b):
    x, y, _ = b; x, y = x.to(device), y.to(device); return crit(backbone(x).float(), y), len(x)
ck = f"{OUT}/checkpoints/backbone_{CFG['backbone']}.pt"
if REUSE_SAVED and os.path.exists(ck):
    s = torch.load(ck, map_location=device, weights_only=False); backbone.load_state_dict(s['state'])
    bb_hist, bb_best = s['hist'], s['best']; print('Loaded saved backbone (set REUSE_SAVED=False to retrain)')
else:
    bb_hist, bb_best = fit(backbone, loader(ds['train'], True), loader(ds_eval['val']), step_bb,
                           CFG['baseline_epochs'], 1e-4, patience=3, name=CFG['backbone'], amp=AMP)
    torch.save({'state': backbone.state_dict(), 'hist': bb_hist, 'best': bb_best}, ck)
plt.figure(figsize=(5, 3.2)); plt.plot(bb_hist['train'], 'o-', label='train'); plt.plot(bb_hist['val'], 'o-', label='validation')
plt.axvline(bb_best, ls='--', c='gray', label='best epoch'); plt.xlabel('epoch'); plt.ylabel('BCE loss'); plt.legend()
plt.title(f"{CFG['backbone']} training"); plt.tight_layout(); plt.savefig(f'{FIG}/backbone_loss.png', dpi=150); plt.show()

In [ ]:
# ===== Cell 14: Extract cloud features (49 tokens + pooled vector) for all splits =====
@torch.no_grad()
def extract(m, d):
    m.eval(); T_, P_, Pr = [], [], []
    for x, _, _ in loader(d, bs=64):
        with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
            tok, pooled = m.tokens(x.to(device))
        tok, pooled = tok.float(), pooled.float()
        T_.append(tok.half().cpu()); P_.append(pooled.cpu()); Pr.append(torch.sigmoid(m.fc(pooled)).cpu())
    return {'tokens': torch.cat(T_), 'pooled': torch.cat(P_), 'probs': torch.cat(Pr).numpy()}
feats = {k: extract(backbone, ds_eval[k]) for k in splits}
baseline_test = feats['test']['probs']
print('Tokens per image:', tuple(feats['test']['tokens'].shape[1:]), '| pooled dim:', feats['test']['pooled'].shape[1])

In [ ]:
# ===== Cell 15: Baseline evaluation on the test set =====
base_auc = per_class_auroc(Y['test'], baseline_test)
print('Baseline mean AUROC — reliable classes: %.3f | all classes: %.3f' % (mean_auroc(Y['test'], baseline_test, reliable),
                                                                            mean_auroc(Y['test'], baseline_test)))
plt.figure(figsize=(7, 4.5))
for c in reliable[:8]:
    i = CLASSES.index(c); fpr, tpr, _ = roc_curve(Y['test'][:, i], baseline_test[:, i]); plt.plot(fpr, tpr, label=f'{c} ({base_auc[c]:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=0.8); plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('Baseline ROC curves (test)'); plt.legend(fontsize=7); plt.tight_layout(); plt.savefig(f'{FIG}/baseline_roc.png', dpi=150); plt.show()

## Part E · Phase 2 — Quantum feature layer and the 16 → 8 qubit decision
The cloud feature vector (1,024 numbers) is compressed by **PCA** to *n* values, scaled to [0, π] and encoded as rotation angles on *n* qubits.
The circuit uses **data re-uploading** (the input is encoded again in every layer, which increases expressivity without more qubits),
trainable RY/RZ rotations and a **ring of CNOTs** that entangles neighbouring qubits. The *n* Pauli-Z expectation values are the quantum features.

**Why 8 qubits and not 16?** Cells 17–18 measure it on this data rather than asserting it: information kept by PCA, how much the circuit's
outputs still vary between patients, gradient variance (barren plateaus), simulation time/memory, and a short head-to-head training run.

In [ ]:
# ===== Cell 16: The quantum circuit (PennyLane) =====
NQ, NL = CFG['n_qubits'], CFG['n_layers']
def make_vqc(n, layers=NL):
    dev = qml.device('default.qubit', wires=n)
    @qml.qnode(dev, interface='torch', diff_method='backprop')
    def circuit(inputs, weights):
        for l in range(layers):
            qml.AngleEmbedding(inputs, wires=range(n), rotation='Y')         # data re-uploading
            for i in range(n):
                qml.RY(weights[l, i, 0], wires=i); qml.RZ(weights[l, i, 1], wires=i)
            for i in range(n):
                qml.CNOT(wires=[i, (i + 1) % n])                              # entangling ring
        return [qml.expval(qml.PauliZ(i)) for i in range(n)]
    return circuit, {'weights': (layers, n, 2)}
quantum_circuit, weight_shapes = make_vqc(NQ)

def fit_angles(n):
    p = PCA(n_components=n, random_state=0).fit(feats['train']['pooled'].numpy())
    s = MinMaxScaler().fit(p.transform(feats['train']['pooled'].numpy()))
    return p, s, {k: torch.tensor(np.clip(s.transform(p.transform(feats[k]['pooled'].numpy())), 0, 1) * np.pi, dtype=torch.float32) for k in splits}
pca, scaler, Q = fit_angles(NQ)
print(f'{NQ} qubits, {NL} layers, {NL * NQ * 2} trainable circuit parameters; PCA keeps {100 * pca.explained_variance_ratio_.sum():.1f}% of variance')
print(qml.draw(quantum_circuit)(Q['train'][0], torch.zeros(NL, NQ, 2)))

In [ ]:
# ===== Cell 17: Qubit study 1: PCA information, output spread, gradient variance, cost =====
qubit_list = [4, 6, 8] if SMOKE else [4, 6, 8, 10, 12, 14, 16]
full_pca = PCA(n_components=min(32, feats['train']['pooled'].shape[0] - 1, feats['train']['pooled'].shape[1]), random_state=0).fit(feats['train']['pooled'].numpy())
cumvar = np.cumsum(full_pca.explained_variance_ratio_)
rng = np.random.default_rng(0); study = []
for n in qubit_list:
    circ, ws = make_vqc(n)
    p_n = PCA(n_components=n, random_state=0).fit(feats['train']['pooled'].numpy())
    s_n = MinMaxScaler().fit(p_n.transform(feats['train']['pooled'].numpy()))
    xa = torch.tensor(np.clip(s_n.transform(p_n.transform(feats['val']['pooled'][:64].numpy())), 0, 1) * np.pi)
    w = torch.tensor(rng.uniform(0, 2 * np.pi, ws['weights']))
    with torch.no_grad():
        out = torch.stack(circ(xa, w), 1)                                   # [64, n] expectation values for 64 patients
    grads = []
    for _ in range(8 if SMOKE else 40):                                     # gradient of <Z0> w.r.t. one parameter, random init
        wr = torch.tensor(rng.uniform(0, 2 * np.pi, ws['weights']), requires_grad=True)
        circ(xa[0], wr)[0].backward(); grads.append(wr.grad[0, 0, 0].item())
    xb = xa[:16].clone(); wt = w.clone().requires_grad_(True); t0 = time.perf_counter()
    torch.stack(circ(xb, wt), 1).sum().backward(); step_s = time.perf_counter() - t0
    study.append({'qubits': n, 'PCA variance kept %': 100 * cumvar[min(n, len(cumvar)) - 1],
                  'output spread between patients (std)': float(out.std(0).mean()), 'gradient variance': float(np.var(grads)),
                  'train step time, batch 16 (s)': step_s, 'state vector size': 2 ** n})
    print(f'  {n:2d} qubits done ({step_s:.2f}s per step)')
qubit_study = pd.DataFrame(study).set_index('qubits')
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
ax[0].plot(range(1, len(cumvar) + 1), 100 * cumvar, '.-'); ax[0].axvline(8, c='r', ls='--'); ax[0].set_title('PCA variance kept'); ax[0].set_xlabel('components = qubits'); ax[0].set_ylabel('%')
ax[1].plot(qubit_study.index, qubit_study.iloc[:, 1], 'o-'); ax[1].set_title('Output spread between patients'); ax[1].set_xlabel('qubits')
ax[2].semilogy(qubit_study.index, qubit_study['gradient variance'], 'o-'); ax[2].set_title('Gradient variance (barren plateau)'); ax[2].set_xlabel('qubits')
ax[3].semilogy(qubit_study.index, qubit_study['train step time, batch 16 (s)'], 'o-'); ax[3].set_title('Training step time (log)'); ax[3].set_xlabel('qubits')
for a in ax[1:]: a.axvline(8, c='r', ls='--')
plt.tight_layout(); plt.savefig(f'{FIG}/qubit_study.png', dpi=150); plt.show()
qubit_study.round(4)

In [ ]:
# ===== Cell 18: Qubit study 2: short head-to-head training, 8 vs 16 qubits =====
class VQCOnly(nn.Module):
    def __init__(self, n):
        super().__init__(); circ, ws = make_vqc(n); self.q = qml.qnn.TorchLayer(circ, ws)
        self.out = nn.Sequential(nn.Linear(n, 32), nn.ReLU(), nn.Linear(32, len(CLASSES)))
    def forward(self, x): return self.out(self.q(x).float())
n_sub = 64 if SMOKE else 512; epochs_sub = 1 if SMOKE else 3
h2h = []
for n in ([8] if SMOKE else [8, 16]):
    _, _, Qn = fit_angles(n); set_seed(0); m = VQCOnly(n)
    opt = torch.optim.Adam(m.parameters(), 1e-2); t0 = time.time()
    for ep in range(epochs_sub):
        for s in range(0, n_sub, 16):
            loss = crit(m(Qn['train'][s:s + 16]), torch.tensor(Y['train'][s:s + 16])); opt.zero_grad(); loss.backward(); opt.step()
    train_s = time.time() - t0
    with torch.no_grad(): pv = torch.sigmoid(m(Qn['val'])).numpy()
    h2h.append({'qubits': n, 'val mean AUROC': mean_auroc(Y['val'], pv), 'training time (s)': train_s,
                'time per epoch on full train set (est., min)': train_s / epochs_sub * len(Y['train']) / n_sub / 60})
head_to_head = pd.DataFrame(h2h).set_index('qubits').round(3)
head_to_head

**Reading the qubit study (fill in with your printed numbers):**
1. **Information:** going from 8 to 16 PCA components adds only a few percent of variance (left plot) — most of the useful signal is already in the first components.
2. **Output spread:** with more qubits the expectation values of different patients bunch together (second plot). Outputs that hardly differ between patients carry little diagnostic signal.
3. **Barren plateau:** gradient variance shrinks roughly exponentially with qubits (third plot, log scale), so a 16-qubit circuit gets almost no training signal.
4. **Cost:** the simulated state has 2ⁿ amplitudes — 65,536 for 16 qubits vs 256 for 8 — so a training step is tens of times slower and memory grows (batch 64 at 16 qubits does not fit in Colab RAM).
5. **Head-to-head:** under the same short budget, 8 qubits trains far faster with equal or better validation AUROC.

**Decision:** 8 qubits is the largest circuit that stays trainable and affordable — trainability over raw information retention.

In [ ]:
# ===== Cell 19: Gradient flow check: the circuit is trainable at 8 qubits =====
w = torch.tensor(np.random.default_rng(1).uniform(0, 2 * np.pi, (NL, NQ, 2)), requires_grad=True)
out = torch.stack(quantum_circuit(Q['train'][:32], w), 1)
out.sum().backward()
print('Expectation values range: [%.3f, %.3f]' % (out.min().item(), out.max().item()))
print('Mean |gradient| over all circuit parameters: %.4f (non-zero -> trainable)' % w.grad.abs().mean().item())

## Part F · Hybrid quantum–classical models vs a classical control
All heads train on the cached cloud features with the **same recipe and three seeds**:

| Model | Description |
|---|---|
| Quantum only | 8 PCA angles → VQC → small classifier |
| Concat hybrid | 1,024 pooled features ‖ 8 quantum features → classifier |
| **Attention hybrid (quantum)** | VQC output is the **query**; the 49 image-region tokens are keys/values; softmax over 49 regions, so the quantum features choose *where* to look |
| Attention + classical control | identical to the attention hybrid, but the VQC is replaced by an 8→8 classical layer |
| Classical head (pooled) | pooled features only, same training recipe |

The **classical control** is the fair comparison: any difference between it and the attention hybrid is due to the quantum circuit itself.

In [ ]:
# ===== Cell 20: Model definitions =====
DIM = backbone.dim
class QuantumOnly(nn.Module):
    def __init__(self):
        super().__init__(); self.quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
        self.classifier = nn.Sequential(nn.Linear(NQ, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, len(CLASSES)))
    def forward(self, tok, pooled, q): return self.classifier(self.quantum_layer(q).float())

class ConcatHybrid(nn.Module):
    def __init__(self):
        super().__init__(); self.quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
        self.classifier = nn.Sequential(nn.Linear(DIM + NQ, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, len(CLASSES)))
    def forward(self, tok, pooled, q): return self.classifier(torch.cat([pooled, self.quantum_layer(q).float()], 1))

class CrossAttentionFusion(nn.Module):
    def __init__(self, d=64):
        super().__init__(); self.q = nn.Linear(NQ, d); self.k = nn.Linear(DIM, d); self.v = nn.Linear(DIM, d); self.s = d ** -0.5
        self.last_weights = None
    def forward(self, tokens, qf):
        w = torch.softmax(self.q(qf)[:, None] @ self.k(tokens).transpose(1, 2) * self.s, dim=-1)   # [B, 1, 49]
        self.last_weights = w.detach(); return (w @ self.v(tokens))[:, 0]

class AttentionHybrid(nn.Module):
    def __init__(self, quantum=True):
        super().__init__()
        self.branch = qml.qnn.TorchLayer(quantum_circuit, weight_shapes) if quantum else nn.Sequential(nn.Linear(NQ, NQ), nn.Tanh())
        self.fusion = CrossAttentionFusion()
        self.classifier = nn.Sequential(nn.Linear(DIM + 64, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, len(CLASSES)))
    def forward(self, tok, pooled, q):
        return self.classifier(torch.cat([pooled, self.fusion(tok.float(), self.branch(q).float())], 1))

class PooledHead(nn.Module):
    def __init__(self):
        super().__init__(); self.classifier = nn.Sequential(nn.Linear(DIM, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, len(CLASSES)))
    def forward(self, tok, pooled, q): return self.classifier(pooled)

BUILDERS = {'Quantum only': QuantumOnly, 'Concat hybrid (quantum)': ConcatHybrid,
            'Attention hybrid (quantum)': lambda: AttentionHybrid(True),
            'Attention + classical control': lambda: AttentionHybrid(False), 'Classical head (pooled)': PooledHead}
pd.DataFrame({k: {'trainable parameters': n_params(b())} for k, b in BUILDERS.items()}).T

In [ ]:
# ===== Cell 21: Train all models (3 seeds each) =====
def head_data(k): return TensorDataset(feats[k]['tokens'], feats[k]['pooled'], Q[k], torch.tensor(Y[k]))
@torch.no_grad()
def predict(h, k):
    h.eval(); return torch.cat([torch.sigmoid(h(t, p, q)) for t, p, q, _ in DataLoader(head_data(k), 256)]).numpy()
test_probs, val_probs, trained, histories = {}, {}, {}, {}
for name, build in BUILDERS.items():
    for seed in CFG['seeds']:
        set_seed(seed); h = build()
        ck = f"{OUT}/checkpoints/head_{name.replace(' ', '_').replace('(', '').replace(')', '').replace('+', 'plus')}_s{seed}.pt"
        if REUSE_SAVED and os.path.exists(ck):
            h.load_state_dict(torch.load(ck)); print(f'Loaded saved {name} (seed {seed})')
        else:
            def step(b, h=h):
                t, p, q, y = b; return crit(h(t, p, q), y), len(y)
            histories[(name, seed)] = fit(h, DataLoader(head_data('train'), 64, shuffle=True), DataLoader(head_data('val'), 256),
                                          step, CFG['head_epochs'], 2e-3, patience=5, name=f'{name} s{seed}')
            torch.save(h.state_dict(), ck)
        trained[(name, seed)] = h; test_probs[(name, seed)] = predict(h, 'test'); val_probs[(name, seed)] = predict(h, 'val')
print('All models ready.')

In [ ]:
# ===== Cell 22: Model comparison with confidence intervals =====
rows = [{'model': f"{CFG['backbone']} baseline", 'mean AUROC (reliable)': mean_auroc(Y['test'], baseline_test, reliable),
         'std over seeds': 0.0, 'mean AUROC (all 14)': mean_auroc(Y['test'], baseline_test), 'parameters (head)': n_params(backbone.fc)}]
ens = {}
for name in BUILDERS:
    r = [mean_auroc(Y['test'], test_probs[(name, s)], reliable) for s in CFG['seeds']]
    ens[name] = np.mean([test_probs[(name, s)] for s in CFG['seeds']], axis=0)          # seed ensemble
    rows.append({'model': name, 'mean AUROC (reliable)': np.mean(r), 'std over seeds': np.std(r),
                 'mean AUROC (all 14)': np.mean([mean_auroc(Y['test'], test_probs[(name, s)]) for s in CFG['seeds']]),
                 'parameters (head)': n_params(trained[(name, CFG['seeds'][0])])})
    rows.append({'model': f'{name} — {len(CFG["seeds"])}-seed ensemble', 'mean AUROC (reliable)': mean_auroc(Y['test'], ens[name], reliable),
                 'std over seeds': np.nan, 'mean AUROC (all 14)': mean_auroc(Y['test'], ens[name]), 'parameters (head)': np.nan})
comparison = pd.DataFrame(rows).set_index('model')
pairs = [('Attention + classical control', 'Attention hybrid (quantum)'), ('baseline', 'Attention hybrid (quantum)'),
         ('baseline', 'Concat hybrid (quantum)'), ('Classical head (pooled)', 'Attention hybrid (quantum)')]
tests = {}
for a, b in pairs:
    pa = baseline_test if a == 'baseline' else ens[a]
    diff, (lo, hi) = paired_bootstrap(Y['test'], pa, ens[b], reliable)
    tests[f'{b}  vs  {a}'] = {'AUROC difference': round(diff, 4), '95% CI low': round(lo, 4), '95% CI high': round(hi, 4),
                             'statistically significant': bool(lo > 0 or hi < 0)}
significance = pd.DataFrame(tests).T
comparison.round(4)

In [ ]:
# ===== Cell 23: Significance tests and per-class AUROC =====
per_class = pd.DataFrame({f"{CFG['backbone']}": per_class_auroc(Y['test'], baseline_test),
                          **{n: per_class_auroc(Y['test'], p) for n, p in ens.items()}})
per_class['test positives'] = [int(Y['test'][:, CLASSES.index(c)].sum()) for c in per_class.index]
per_class['reliable'] = [c in reliable for c in per_class.index]
plt.figure(figsize=(11, 4))
plot_df = per_class.loc[[c for c in reliable if c in per_class.index], [f"{CFG['backbone']}", 'Attention + classical control', 'Attention hybrid (quantum)']]
plot_df.plot.bar(ax=plt.gca(), width=0.8); plt.ylim(0.4, 1.0); plt.ylabel('AUROC'); plt.title('Per-class AUROC (test, reliable classes)')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.savefig(f'{FIG}/per_class_auroc.png', dpi=150); plt.show()
display_sig = significance
print(significance.to_string())
per_class.round(3)

## Part G · Phase 3 — Fog-tier triage (MobileNetV3-Small)
The fog node runs a small CNN on a CPU-class device and answers one question: *does this X-ray need the cloud?*
Its threshold is chosen on **validation** to reach a target recall (a missed abnormal case costs more than an extra cloud call); results are reported on **test**.

In [ ]:
# ===== Cell 24: Train the fog triage model =====
class TriageNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.features = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None).features
        self.drop = nn.Dropout(0.2); self.fc = nn.Linear(576, 1)
    def forward(self, x): return self.fc(self.drop(self.features(x).mean(dim=(2, 3)))).squeeze(1)
set_seed(42)
triage_model = TriageNet(CFG['pretrained']).to(device)
pw = torch.tensor((1 - YA['train'].mean()) / max(YA['train'].mean(), 1e-6), device=device)
crit_t = nn.BCEWithLogitsLoss(pos_weight=pw)
def step_t(b):
    x, _, a = b; x, a = x.to(device), a.to(device); return crit_t(triage_model(x).float(), a), len(x)
ck = f'{OUT}/checkpoints/triage_mobilenetv3s.pt'
if REUSE_SAVED and os.path.exists(ck):
    triage_model.load_state_dict(torch.load(ck, map_location=device)); print('Loaded saved triage model')
else:
    fit(triage_model, loader(ds['train'], True, 64), loader(ds_eval['val'], bs=64), step_t, CFG['triage_epochs'], 3e-4,
        patience=4, name='Fog triage', amp=AMP)
    torch.save(triage_model.state_dict(), ck)
@torch.no_grad()
def triage_probs(k):
    triage_model.eval(); return torch.cat([torch.sigmoid(triage_model(x.to(device)).float()).cpu() for x, _, _ in loader(ds_eval[k], bs=64)]).numpy()
tp = {k: triage_probs(k) for k in ('val', 'test')}
size_mb = lambda m: sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6
print('Triage AUROC — val %.3f | test %.3f' % (roc_auc_score(YA['val'], tp['val']), roc_auc_score(YA['test'], tp['test'])))
print(f'Model size: triage {size_mb(triage_model):.1f} MB vs cloud backbone {size_mb(backbone):.1f} MB '
      f'({n_params(backbone) / n_params(triage_model):.1f}x more parameters in the cloud)')

In [ ]:
# ===== Cell 25: Choose the fog threshold on validation, report on test =====
def tstats(p, y, t):
    send = p > t; yb = y == 1
    return {'recall (abnormal sent to cloud)': (send & yb).sum() / max(yb.sum(), 1),
            'specificity (normals kept at fog)': (~send & ~yb).sum() / max((~yb).sum(), 1),
            'precision': (send & yb).sum() / max(send.sum(), 1), 'sent to cloud %': 100 * send.mean()}
sweep = pd.DataFrame({t: tstats(tp['val'], YA['val'], t) for t in np.round(np.arange(0.05, 0.80, 0.05), 2)}).T
ok = sweep[sweep['recall (abnormal sent to cloud)'] >= CFG['triage_target_recall']]
THRESHOLD = float(ok.index.max()) if len(ok) else float(sweep.index.min())
fog_test = tstats(tp['test'], YA['test'], THRESHOLD)
cm = confusion_matrix(YA['test'], tp['test'] > THRESHOLD, labels=[0, 1])
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
fpr, tpr, _ = roc_curve(YA['test'], tp['test']); ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
ax[0].set_title(f"Triage ROC (test AUROC {roc_auc_score(YA['test'], tp['test']):.2f})"); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[1].plot(sweep.index, sweep['recall (abnormal sent to cloud)'], 'o-', label='recall'); ax[1].plot(sweep.index, sweep['sent to cloud %'] / 100, 's-', label='share sent to cloud')
ax[1].axvline(THRESHOLD, c='r', ls='--', label=f'chosen {THRESHOLD}'); ax[1].set_xlabel('threshold'); ax[1].legend(); ax[1].set_title('Validation threshold sweep')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[2], xticklabels=['kept at fog', 'sent to cloud'], yticklabels=['normal', 'abnormal'])
ax[2].set_title('Test confusion matrix'); plt.tight_layout(); plt.savefig(f'{FIG}/fog_triage.png', dpi=150); plt.show()
print(f'Chosen threshold (from validation): {THRESHOLD}')
pd.Series(fog_test).round(3).to_frame('test')

In [ ]:
# ===== Cell 26: Fog vs cloud compute: CPU speed on a 1-thread "fog device" =====
def time_model(m, dev_, n=20):
    m = copy.deepcopy(m).to(dev_).eval().float(); x = torch.randn(1, 3, 224, 224, device=dev_)
    with torch.no_grad():
        for _ in range(3): m(x)
        if dev_.type == 'cuda': torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(n): m(x)
        if dev_.type == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000
threads = torch.get_num_threads(); torch.set_num_threads(1)
speed = {'Fog: MobileNetV3-Small': {'CPU 1-thread ms': time_model(triage_model, torch.device('cpu'))},
         f"Cloud: {CFG['backbone']}": {'CPU 1-thread ms': time_model(backbone, torch.device('cpu'))}}
torch.set_num_threads(threads)
if device.type == 'cuda':
    for k, m in zip(speed, (triage_model, backbone)): speed[k]['GPU ms'] = time_model(m, device)
speed = pd.DataFrame(speed).T.round(1)
SPEEDUP = speed.iloc[1, 0] / speed.iloc[0, 0]
print(f'On the fog CPU the triage model is {SPEEDUP:.1f}x faster than running the cloud backbone there.'); speed

In [ ]:
# ===== Cell 27: Expected cloud savings at realistic screening prevalence =====
rec, spec = fog_test['recall (abnormal sent to cloud)'], fog_test['specificity (normals kept at fog)']
prev = pd.DataFrame([{'abnormal share in population': (f'{int(100 * a)}%' if j else f'this test set ({100 * a:.0f}%)'),
                      'sent to cloud %': 100 * (a * rec + (1 - a) * (1 - spec)),
                      'cloud calls avoided %': 100 * (1 - (a * rec + (1 - a) * (1 - spec)))}
                     for j, a in enumerate((YA['test'].mean(), 0.30, 0.20, 0.10))]).set_index('abnormal share in population').round(1)
print('This sample is ~half abnormal; screening populations are mostly normal, where the fog avoids many more cloud calls '
      '(same recall/specificity, re-weighted).'); prev

## Part H · Fog ↔ cloud integration (running system)
Two tiers run as separate components that talk over **HTTP**:

```
 X-ray device ──PNG──▶ FOG NODE (CPU, 1 thread)                       CLOUD SERVICE (FastAPI, port 8001)
                        preprocess → MobileNetV3 triage                 /diagnose: backbone → PCA → 8-qubit VQC
                        p ≤ threshold → keep locally ("low-risk")        → quantum-attention fusion → 14 findings
                        p > threshold → JPEG ──emulated link──────────▶  → Grad-CAM++ + reasoning report
                        cloud unreachable → fallback: flag for review  ◀── JSON report ── emulated link
```
The network between them is emulated (latency + bandwidth), so the same code shows behaviour on good, average and poor links.

In [ ]:
# ===== Cell 28: Cloud model: backbone + frozen PCA + quantum attention hybrid (seed ensemble) =====
class FrozenPCAScale(nn.Module):
    """PCA + [0, pi] scaling as fixed torch ops, so gradients flow image -> backbone -> PCA -> quantum circuit."""
    def __init__(self, p, s):
        super().__init__()
        self.register_buffer('mean', torch.tensor(p.mean_, dtype=torch.float32)); self.register_buffer('comp', torch.tensor(p.components_, dtype=torch.float32))
        self.register_buffer('dmin', torch.tensor(s.data_min_, dtype=torch.float32))
        self.register_buffer('drng', torch.tensor(np.where(s.data_range_ == 0, 1, s.data_range_), dtype=torch.float32))
    def forward(self, x): return torch.clamp(((x - self.mean) @ self.comp.T - self.dmin) / self.drng, 0, 1) * np.pi

class CloudModel(nn.Module):
    def __init__(self, bb, heads):
        super().__init__(); self.backbone, self.pca, self.heads = bb, FrozenPCAScale(pca, scaler), nn.ModuleList(heads)
    def forward(self, x):
        tok, pooled = self.backbone.tokens(x); q = self.pca(pooled)
        return torch.stack([h(tok, pooled, q) for h in self.heads]).mean(0)
    def attention_map(self):
        return torch.stack([h.fusion.last_weights for h in self.heads]).mean(0)[:, 0]     # [B, 49]

bb_cpu = copy.deepcopy(backbone).cpu().float().eval()
cloud_model = CloudModel(bb_cpu, [trained[('Attention hybrid (quantum)', s)] for s in CFG['seeds']]).eval()
x8 = torch.stack([ds_eval['test'][i][0] for i in range(min(8, len(ds_eval['test'])))])
with torch.no_grad(): e2e = torch.sigmoid(cloud_model(x8)).numpy()
print('End-to-end cloud model vs cached-feature predictions, max difference: %.2e' % np.abs(e2e - ens['Attention hybrid (quantum)'][:len(x8)]).max())
torch.save(cloud_model.state_dict(), f'{OUT}/checkpoints/cloud_model_full.pt')

In [ ]:
# ===== Cell 29: Explainable-AI reasoning engine (used by the cloud service) =====
# Image zones on the 224x224 frontal view. Radiological convention: image-left = patient's RIGHT side.
H = W = 224; yy, xx = np.mgrid[0:H, 0:W] / H
BORDER = (xx < 0.08) | (xx > 0.92) | (yy < 0.08) | (yy > 0.92)
CENTRAL = (xx >= 0.42) & (xx <= 0.58) & ~BORDER
ZONES = {}
for side, cond in (('right', xx < 0.42), ('left', xx > 0.58)):
    for lvl, (a, b) in (('upper', (0.08, 0.37)), ('middle', (0.37, 0.63)), ('lower', (0.63, 0.92))):
        ZONES[f'{side} {lvl} zone'] = cond & (yy >= a) & (yy < b) & ~BORDER
ZONES['central (heart / mediastinum)'] = CENTRAL
LUNG = [z for z in ZONES if 'zone' in z]
EXPECTED = {   # where each finding typically appears on a frontal chest X-ray (simplified, for a plausibility check)
    'Atelectasis': [z for z in LUNG if 'lower' in z or 'middle' in z], 'Cardiomegaly': ['central (heart / mediastinum)'] + [z for z in LUNG if 'lower' in z],
    'Effusion': [z for z in LUNG if 'lower' in z], 'Infiltration': LUNG, 'Mass': LUNG, 'Nodule': LUNG, 'Pneumonia': LUNG,
    'Pneumothorax': [z for z in LUNG if 'upper' in z or 'middle' in z], 'Consolidation': LUNG,
    'Edema': [z for z in LUNG if 'middle' in z] + ['central (heart / mediastinum)'], 'Emphysema': [z for z in LUNG if 'upper' in z or 'middle' in z],
    'Fibrosis': [z for z in LUNG if 'lower' in z or 'middle' in z], 'Pleural_Thickening': LUNG,
    'Hernia': ['central (heart / mediastinum)'] + [z for z in LUNG if 'lower' in z]}
TYPICAL = {'Atelectasis': 'collapsed lung tissue, often in the lower/middle zones', 'Cardiomegaly': 'an enlarged heart shadow in the central lower chest',
           'Effusion': 'fluid collecting at the lung bases (costophrenic angles)', 'Infiltration': 'patchy opacities anywhere in the lung fields',
           'Mass': 'a focal opacity > 3 cm within the lungs', 'Nodule': 'a small focal opacity within the lungs', 'Pneumonia': 'airspace opacity, often lower zones',
           'Pneumothorax': 'air at the lung apex / periphery', 'Consolidation': 'dense airspace filling of a lung region',
           'Edema': 'bilateral perihilar haziness around the heart', 'Emphysema': 'hyperinflated, darker upper lung zones',
           'Fibrosis': 'reticular scarring, often peripheral and lower', 'Pleural_Thickening': 'thickened pleura along the lung margins',
           'Hernia': 'abdominal content above the diaphragm, low central chest'}

def zone_shares(cam):
    tot = cam.sum() + 1e-8
    return {z: float((cam * m).sum() / tot) for z, m in ZONES.items()} | {'image border': float((cam * BORDER).sum() / tot)}

def occlusion_test(model, x, cls, cam, frac=0.15):
    """Remove the most-highlighted 15% of pixels and measure how much the probability drops (faithfulness)."""
    thr = np.quantile(cam, 1 - frac); m = torch.tensor(cam >= thr)
    xo = x.clone(); xo[:, :, m] = 0.0                       # 0 = mean intensity after normalisation
    with torch.no_grad(): return float(torch.sigmoid(model(xo))[0, cls])

def reasoning(cls_name, prob, cam, prob_after, att_map=None):
    sh = zone_shares(cam); top_zone = max([z for z in ZONES], key=lambda z: sh[z])
    exp = EXPECTED[cls_name]; exp_share = sum(sh[z] for z in exp)
    exp_area = sum(ZONES[z].mean() for z in exp) / (1 - BORDER.mean()); lift = exp_share / max(exp_area, 1e-6)
    conf = 'high' if prob >= 0.7 else 'moderate' if prob >= 0.5 else 'low'
    lines = [f'**Finding: {cls_name}** — probability {prob:.2f} ({conf} confidence).',
             f'**Where the model looked:** strongest attention in the **{top_zone}** ({100 * sh[top_zone]:.0f}% of heat). '
             f'{cls_name} typically shows {TYPICAL[cls_name]}.']
    if sh['image border'] >= 0.30:
        verdict = 'possible shortcut'
        lines.append(f'⚠ **Shortcut warning:** {100 * sh["image border"]:.0f}% of the heat lies on the image border (labels, markers, edges), '
                     'not on anatomy. Treat this prediction with caution.')
    elif lift >= 1.15:
        verdict = 'anatomically plausible'
        lines.append(f'✔ **Plausibility:** {100 * exp_share:.0f}% of the heat falls in the zones where {cls_name} is expected '
                     f'({lift:.1f}x more than chance) — the attention is anatomically plausible.')
    else:
        verdict = 'diffuse / uncertain'
        lines.append(f'• **Plausibility:** attention is diffuse ({100 * exp_share:.0f}% in expected zones, {lift:.1f}x chance) — '
                     'the model has not localised a clear region.')
    drop = prob - prob_after
    lines.append(f'**Faithfulness (occlusion test):** hiding the highlighted region changes the probability from {prob:.2f} to {prob_after:.2f} '
                 + ('— the highlighted region genuinely drives the prediction.' if drop > 0.05 else '— small change; the prediction relies on wider context.'))
    if att_map is not None:
        g = att_map.reshape(7, 7); r, c = np.unravel_index(int(np.argmax(g)), g.shape)
        side = "patient's right" if c < 3 else "patient's left" if c > 3 else 'centre'
        lvl = 'upper' if r < 2 else 'lower' if r > 4 else 'middle'
        lines.append(f'**Quantum attention:** the 8-qubit query attended most to the {lvl} {side} region of the image.')
    lines.append(('Recommendation: radiologist review advised.' if conf != 'high' or verdict != 'anatomically plausible'
                  else 'Recommendation: finding is confident and well-localised; confirm in routine reporting.'))
    return '\n\n'.join(lines), {'verdict': verdict, 'top_zone': top_zone, 'expected_zone_share': exp_share, 'border_share': sh['image border'],
                                'prob_after_occlusion': prob_after, 'zone_shares': sh}
print('Reasoning engine ready:', len(ZONES), 'anatomical zones,', len(EXPECTED), 'finding-specific expectations')

In [ ]:
# ===== Cell 30: Cloud service (FastAPI) running in the background =====
import uvicorn, requests
from fastapi import FastAPI, Request
cloud_cam = GradCAMPlusPlus(model=cloud_model, target_layers=[cloud_model.backbone.cam_layer])
cloud_lock = threading.Lock()

def png_b64(arr):
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format='PNG'); return base64.b64encode(buf.getvalue()).decode()

def cloud_diagnose(gray, explain=True):
    with cloud_lock:
        t = {}; x = to_input(gray)[None]; t0 = time.perf_counter()
        with torch.no_grad(): probs = torch.sigmoid(cloud_model(x))[0].numpy(); att = cloud_model.attention_map()[0].numpy()
        t['diagnosis (backbone + 8-qubit VQC + attention)'] = (time.perf_counter() - t0) * 1000
        top = np.argsort(-probs)[:3]
        out = {'findings': [(CLASSES[i], float(probs[i])) for i in top], 'probs': {c: float(v) for c, v in zip(CLASSES, probs)}}
        if explain:
            t0 = time.perf_counter(); c = int(top[0])
            cam = cloud_cam(input_tensor=x, targets=[ClassifierOutputTarget(c)])[0]
            after = occlusion_test(cloud_model, x, c, cam)
            text, info = reasoning(CLASSES[c], float(probs[c]), cam, after, att)
            img = denormalize(x[0])
            att_img = F.interpolate(torch.tensor(att).reshape(1, 1, 7, 7), size=(224, 224), mode='bilinear')[0, 0].numpy()
            att_img = (att_img - att_img.min()) / (att_img.max() - att_img.min() + 1e-8)
            out.update({'heatmap_png': png_b64(show_cam_on_image(img, cam, use_rgb=True)),
                        'quantum_attention_png': png_b64(show_cam_on_image(img, att_img, use_rgb=True)),
                        'reasoning': text, 'xai': {k: v for k, v in info.items() if k != 'zone_shares'}, 'zone_shares': info['zone_shares']})
            t['explanation (Grad-CAM++ + reasoning)'] = (time.perf_counter() - t0) * 1000
        out['timings_ms'] = t
        return out

cloud_app = FastAPI(title='QFog cloud tier')
@cloud_app.get('/health')
def health(): return {'ok': True, 'model': f"{CFG['backbone']} + {NQ}-qubit VQC attention hybrid"}
@cloud_app.post('/diagnose')
async def diagnose(request: Request):
    body = await request.body(); explain = request.query_params.get('explain', '1') == '1'
    return cloud_diagnose(load_gray(Image.open(io.BytesIO(body))), explain)

CLOUD_PORT = 8001; CLOUD_URL = f'http://127.0.0.1:{CLOUD_PORT}'
def cloud_up():
    try: return requests.get(f'{CLOUD_URL}/health', timeout=1).ok
    except Exception: return False
if not cloud_up():
    server = uvicorn.Server(uvicorn.Config(cloud_app, host='127.0.0.1', port=CLOUD_PORT, log_level='warning'))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(60):
        if cloud_up(): break
        time.sleep(0.5)
print('Cloud service running:', cloud_up(), '|', requests.get(f'{CLOUD_URL}/health').json())

In [ ]:
# ===== Cell 31: Fog node (triage on CPU + emulated network + fallback) =====
triage_cpu = copy.deepcopy(triage_model).cpu().float().eval()
def emulate_link(nbytes, latency_ms, mbps):
    d = latency_ms / 1000 + nbytes * 8 / (mbps * 1e6); time.sleep(d); return d * 1000

def fog_node(raw_png_bytes, mode='fog+cloud', latency_ms=40, mbps=10, cloud_down=False, explain=True):
    """mode 'fog+cloud': triage first; 'cloud-only': forward everything (the baseline without a fog tier)."""
    rec, t = {'mode': mode, 'device→fog bytes': len(raw_png_bytes)}, {}
    t_all = time.perf_counter(); t0 = time.perf_counter()
    gray = load_gray(Image.open(io.BytesIO(raw_png_bytes))); t['fog preprocess'] = (time.perf_counter() - t0) * 1000
    if mode == 'fog+cloud':
        t0 = time.perf_counter()
        with torch.no_grad(): p = torch.sigmoid(triage_cpu(to_input(gray)[None])).item()
        t['fog triage (MobileNetV3, CPU)'] = (time.perf_counter() - t0) * 1000
        rec['p_abnormal'] = p; send = p > THRESHOLD
    else:
        rec['p_abnormal'] = None; send = True
    rec['sent_to_cloud'] = send
    if not send:
        rec.update(decision='LOW-RISK on the fog screen — not reviewed by the cloud model. Screening result only: '
                       'this is not a diagnosis and not a clearance.', uplink_bytes=0)
    else:
        buf = io.BytesIO(); Image.fromarray(gray).save(buf, format='JPEG', quality=90); payload = buf.getvalue()
        rec['uplink_bytes'] = len(payload)
        try:
            if cloud_down: raise ConnectionError('cloud unreachable (simulated outage)')
            up = emulate_link(len(payload), latency_ms, mbps); t0 = time.perf_counter()
            r = requests.post(f'{CLOUD_URL}/diagnose', data=payload, params={'explain': int(explain)}, timeout=120); r.raise_for_status()
            t['cloud round trip (compute)'] = (time.perf_counter() - t0) * 1000
            down = emulate_link(len(r.content), latency_ms, mbps); t['network (emulated)'] = up + down
            rec.update(decision='ABNORMAL — flagged by fog, diagnosed in the cloud' if mode == 'fog+cloud' else 'Diagnosed in the cloud',
                       cloud=r.json(), downlink_bytes=len(r.content))
        except Exception as e:
            rec.update(decision=f'FALLBACK — {e}. Fog triage says p(abnormal)={rec["p_abnormal"] if rec["p_abnormal"] is not None else float("nan"):.2f}; '
                                'case queued and flagged for radiologist review.', error=str(e))
    t['total'] = (time.perf_counter() - t_all) * 1000; rec['timings_ms'] = t
    return rec
def png_bytes(name):
    with open(img_path[name], 'rb') as f: return f.read()
r = fog_node(png_bytes(splits['test']['Image Index'][0]))
print('Fog node test call ->', r['decision'], '| total %.0f ms' % r['timings_ms']['total'])

In [ ]:
# ===== Cell 32: System experiment: fog+cloud vs cloud-only, network conditions, outage =====
test_df = splits['test']
N_SYS = 10 if SMOKE else 300
sample = test_df.sample(min(N_SYS, len(test_df)), random_state=0)
def replay(df_, **kw):
    out = []
    for _, row in df_.iterrows():
        rr = fog_node(png_bytes(row['Image Index']), **kw)
        out.append({'abnormal': int(row['abnormal']), 'sent_to_cloud': rr['sent_to_cloud'], 'uplink_bytes': rr['uplink_bytes'],
                    'total_ms': rr['timings_ms']['total'], 'cloud_ms': rr['timings_ms'].get('cloud round trip (compute)', 0.0),
                    'fallback': 'FALLBACK' in rr['decision']})
    return pd.DataFrame(out)
runs = {}
for mode in ('fog+cloud', 'cloud-only'):
    runs[f'{mode} | 40 ms, 10 Mbps'] = replay(sample, mode=mode, latency_ms=40, mbps=10, explain=True)
small = sample.head(5 if SMOKE else 60)
for lat, mb, tag in ((10, 50, 'good link 10 ms, 50 Mbps'), (150, 1, 'poor link 150 ms, 1 Mbps')):
    for mode in ('fog+cloud', 'cloud-only'):
        runs[f'{mode} | {tag}'] = replay(small, mode=mode, latency_ms=lat, mbps=mb, explain=True)
runs['fog+cloud | cloud outage'] = replay(small, mode='fog+cloud', cloud_down=True)
def summarise(g):
    abn = g['abnormal'] == 1
    return pd.Series({'images': len(g), 'sent to cloud %': 100 * g['sent_to_cloud'].mean(),
                      'abnormal reaching cloud %': 100 * g.loc[abn, 'sent_to_cloud'].mean() if abn.any() else np.nan,
                      'MB uploaded': g['uplink_bytes'].sum() / 1e6, 'cloud compute s': g['cloud_ms'].sum() / 1000,
                      'mean latency ms': g['total_ms'].mean(), 'p95 latency ms': g['total_ms'].quantile(0.95), 'fallbacks handled': int(g['fallback'].sum())})
system = pd.DataFrame({k: summarise(v) for k, v in runs.items()}).T.round(2)
base, fogc = system.loc['cloud-only | 40 ms, 10 Mbps'], system.loc['fog+cloud | 40 ms, 10 Mbps']
SAVINGS = {'cloud calls avoided %': 100 - fogc['sent to cloud %'], 'upload saved %': 100 * (1 - fogc['MB uploaded'] / max(base['MB uploaded'], 1e-9)),
           'cloud compute saved %': 100 * (1 - fogc['cloud compute s'] / max(base['cloud compute s'], 1e-9))}
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, col in zip(ax, ['sent to cloud %', 'cloud compute s', 'mean latency ms']):
    system.loc[[k for k in system.index if 'outage' not in k], col].plot.barh(ax=a, color=['#4c72b0', '#dd8452'] * 3); a.set_title(col)
plt.tight_layout(); plt.savefig(f'{FIG}/system_results.png', dpi=150); plt.show()
print({k: round(v, 1) for k, v in SAVINGS.items()})
system

## Part I · Explainable AI with reasoning
Four complementary explanations, each answering a different clinician question:

| Question | Method |
|---|---|
| *Where did the model look?* | **Grad-CAM++ through the full hybrid** — gradients pass through the quantum circuit, so the heatmap explains the model that actually made the diagnosis |
| *What did the quantum part focus on?* | **Quantum attention map** — the 8-qubit query's attention weights over the 49 image regions |
| *Does that make anatomical sense?* | **Zone analysis + plausibility check** — heat share per lung zone vs where the finding typically appears; **shortcut warning** if heat sits on the border |
| *Is the explanation faithful?* | **Occlusion test** — hide the highlighted pixels and measure the probability drop (vs random pixels) |

In [ ]:
# ===== Cell 33: Grad-CAM++: baseline vs hybrid (through the quantum circuit) vs quantum attention =====
base_cam = GradCAMPlusPlus(model=bb_cpu, target_layers=[bb_cpu.cam_layer])
hp = ens['Attention hybrid (quantum)']
cand = sorted([(i, c) for i in range(len(hp)) for c in range(len(CLASSES)) if Y['test'][i, c] == 1 and CLASSES[c] in reliable],
              key=lambda t: -hp[t])[:5] or [(0, int(np.argmax(hp[0])))]
fig, ax = plt.subplots(4, len(cand), figsize=(3.2 * len(cand), 13), squeeze=False)
for j, (i, c) in enumerate(cand):
    x = ds_eval['test'][i][0][None]; img = denormalize(x[0])
    cb = base_cam(input_tensor=x, targets=[ClassifierOutputTarget(c)])[0]
    ch = cloud_cam(input_tensor=x, targets=[ClassifierOutputTarget(c)])[0]
    with torch.no_grad(): cloud_model(x)
    att = F.interpolate(cloud_model.attention_map().reshape(1, 1, 7, 7), size=(224, 224), mode='bilinear')[0, 0].numpy()
    att = (att - att.min()) / (att.max() - att.min() + 1e-8)
    for k, (im, ttl) in enumerate([(img, f'{CLASSES[c]} (p={hp[i, c]:.2f})'), (show_cam_on_image(img, cb, use_rgb=True), 'Grad-CAM++ baseline'),
                                    (show_cam_on_image(img, ch, use_rgb=True), 'Grad-CAM++ hybrid (via VQC)'), (show_cam_on_image(img, att, use_rgb=True), 'Quantum attention')]):
        ax[k, j].imshow(im); ax[k, j].set_title(ttl, fontsize=9); ax[k, j].axis('off')
plt.tight_layout(); plt.savefig(f'{FIG}/xai_gradcam_quantum_attention.png', dpi=140); plt.show()

In [ ]:
# ===== Cell 34: Reasoning report for one case =====
i, c = cand[0]
res = cloud_diagnose(imgs['test'][i])
print('Ground truth:', test_df.loc[i, 'Finding Labels']); print()
print(res['reasoning'].replace('**', ''))
zs = pd.Series(res['zone_shares']).sort_values(ascending=False) * 100
plt.figure(figsize=(7, 3)); zs.plot.barh(color='indianred'); plt.xlabel('% of heatmap heat'); plt.title('Where the heat falls (anatomical zones)')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.savefig(f'{FIG}/xai_zone_shares.png', dpi=140); plt.show()

In [ ]:
# ===== Cell 35: Global XAI: plausibility, shortcut learning and faithfulness over test images =====
rng = np.random.default_rng(0)
idx = rng.choice(len(test_df), min(len(test_df), 12 if SMOKE else 100), replace=False)
rows = []
for i in idx:
    x = ds_eval['test'][i][0][None]
    with torch.no_grad(): pr = torch.sigmoid(cloud_model(x))[0].numpy()
    c = int(np.argmax(pr)); cam = cloud_cam(input_tensor=x, targets=[ClassifierOutputTarget(c)])[0]
    after = occlusion_test(cloud_model, x, c, cam)
    rnd = rng.random(cam.shape); after_rand = occlusion_test(cloud_model, x, c, rnd)
    _, info = reasoning(CLASSES[c], float(pr[c]), cam, after)
    rows.append({'class': CLASSES[c], 'verdict': info['verdict'], 'border share': info['border_share'],
                 'expected-zone share': info['expected_zone_share'], 'drop (highlighted removed)': pr[c] - after, 'drop (random removed)': pr[c] - after_rand})
xai_df = pd.DataFrame(rows)
xai_summary = pd.Series({'images analysed': len(xai_df),
                         'anatomically plausible %': 100 * (xai_df['verdict'] == 'anatomically plausible').mean(),
                         'possible shortcut %': 100 * (xai_df['verdict'] == 'possible shortcut').mean(),
                         'diffuse / uncertain %': 100 * (xai_df['verdict'] == 'diffuse / uncertain').mean(),
                         'mean heat on border %': 100 * xai_df['border share'].mean(),
                         'prob. drop, highlighted pixels removed': xai_df['drop (highlighted removed)'].mean(),
                         'prob. drop, random pixels removed': xai_df['drop (random removed)'].mean()}).round(3)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
xai_df['verdict'].value_counts().plot.bar(ax=ax[0], color=['#55a868', '#c44e52', '#8172b2'][:xai_df['verdict'].nunique()]); ax[0].set_title('Explanation verdicts'); ax[0].tick_params(axis='x', rotation=0)
ax[1].bar(['highlighted\nregion removed', 'random pixels\nremoved'], [xai_summary.iloc[5], xai_summary.iloc[6]], color=['#c44e52', 'gray'])
ax[1].set_title('Faithfulness: probability drop'); plt.tight_layout(); plt.savefig(f'{FIG}/xai_global.png', dpi=140); plt.show()
xai_summary.to_frame('value')

In [ ]:
# ===== Cell 36: Quantum feature attribution: which qubits drive each finding =====
h = trained[('Attention hybrid (quantum)', CFG['seeds'][0])]; h.eval()
n_attr = min(64, len(Q['test']))
q = Q['test'][:n_attr].clone().requires_grad_(True)
logits = h(feats['test']['tokens'][:n_attr], feats['test']['pooled'][:n_attr], q)
attr = np.zeros((len(reliable), NQ))
for k, c in enumerate(reliable):
    q.grad = None; logits[:, CLASSES.index(c)].sum().backward(retain_graph=True)
    attr[k] = (q.grad * q).abs().mean(0).detach().numpy()                     # gradient x input
attr = attr / (attr.sum(1, keepdims=True) + 1e-12)
plt.figure(figsize=(8, 0.35 * len(reliable) + 1.5)); sns.heatmap(pd.DataFrame(attr, index=reliable, columns=[f'q{i}' for i in range(NQ)]), cmap='viridis', annot=True, fmt='.2f')
plt.title('Relative influence of each qubit input (gradient × input)'); plt.tight_layout(); plt.savefig(f'{FIG}/xai_qubit_attribution.png', dpi=140); plt.show()

In [ ]:
# ===== Cell 37: Shortcut-learning check on the baseline vs hybrid =====
sub = idx[:min(len(idx), 40)]
sc = {}
for name, cam_obj, m in (('Baseline', base_cam, bb_cpu), ('Quantum hybrid', cloud_cam, cloud_model)):
    b = []
    for i in sub:
        x = ds_eval['test'][i][0][None]
        with torch.no_grad(): c = int(torch.sigmoid(m(x))[0].argmax())
        b.append(zone_shares(cam_obj(input_tensor=x, targets=[ClassifierOutputTarget(c)])[0])['image border'])
    sc[name] = {'mean heat on border %': 100 * np.mean(b), 'images with >30% border heat %': 100 * np.mean(np.array(b) > 0.30)}
shortcut = pd.DataFrame(sc).T.round(1)
print(f'Border band = outer 8% of the image, i.e. {100 * BORDER.mean():.0f}% of its area (uniform heat would put {100 * BORDER.mean():.0f}% there).')
shortcut

## Part I-b · Evaluation hardening (added after review)
Four checks that make the claims safer to defend. None of them retrains the cloud model or changes any earlier result.

| Cell | Question it answers |
|---|---|
| **37a** | What does it cost (cloud calls avoided, abnormal images still kept at the fog) to raise the fog's recall target from 85% to 90% and 95%? Thresholds are chosen on **validation**; recall, specificity and cloud-call rate are reported on **test**. |
| **37b** | How stable are the latency and savings numbers? One fixed workload, repeated passes, latency **distributions** (p50 / p95 / p99), and every component timed separately: fog inference, network delay, upload, cloud inference, download. |
| **37c** | Do image borders (markers, labels, edges) carry predictive information? A frame-only baseline, an interior-only model, and controlled perturbations of the cloud model's input. |

**Measured vs emulated.** Fog inference and cloud inference are measured on this Colab machine (fog and cloud run on the same host). Network delay, upload and download times are **emulated**: they are computed from payload size, latency and bandwidth (no packet loss, no jitter).

**Safety wording.** The fog node never declares an X-ray "normal" or "safe". A case kept at the fog is only *low-risk on the fog screen*: it was **not reviewed by the cloud model**, and some of those cases are abnormal. Cell 37a counts them.

In [ ]:
# ===== Cell 37a: Fog operating points - thresholds chosen on VALIDATION for 85 / 90 / 95% recall, reported on TEST =====
from IPython.display import display
def wilson(k, n, z=1.96):
    """95% Wilson interval for a proportion k / n."""
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z ** 2 / n; c = p + z ** 2 / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2))
    return (c - h) / d, (c + h) / d

GRID = np.round(np.arange(0.005, 0.90, 0.005), 3)          # finer than Cell 25, so a 95% target can be reached
yv, yt = YA['val'] == 1, YA['test'] == 1
def pick_threshold(target):
    """Largest validation threshold whose validation recall >= target (= fewest cloud calls that still meet the target)."""
    ok = [t for t in GRID if (tp['val'][yv] > t).mean() >= target]
    return (float(max(ok)), True) if ok else (float(GRID.min()), False)

rows, THRESHOLDS_BY_TARGET, missed = [], {}, {}
for target in (0.85, 0.90, 0.95):
    t, met = pick_threshold(target); THRESHOLDS_BY_TARGET[target] = t
    send = tp['test'] > t
    tpn, fnn = int((send & yt).sum()), int((~send & yt).sum()); tnn, fpn = int((~send & ~yt).sum()), int((send & ~yt).sum())
    lo, hi = wilson(tpn, tpn + fnn); rec_, spec_ = tpn / max(tpn + fnn, 1), tnn / max(tnn + fpn, 1)
    rows.append({'recall target (validation)': f'{target:.0%}', 'threshold': t, 'target reachable on validation': met,
                 'validation recall': float((tp['val'][yv] > t).mean()),
                 'test recall': rec_, 'test recall 95% CI': f'[{lo:.3f}, {hi:.3f}]', 'test recall >= target': bool(rec_ >= target),
                 'abnormal kept at fog (n)': fnn, 'abnormal kept at fog (%)': 100 * fnn / max(tpn + fnn, 1),
                 'test specificity': spec_, 'test precision': tpn / max(tpn + fpn, 1),
                 'cloud-call rate % (this test set)': 100 * float(send.mean()), 'cloud calls avoided % (this test set)': 100 * (1 - float(send.mean())),
                 'cloud calls avoided % at 20% abnormal (projection)': 100 * (1 - (0.2 * rec_ + 0.8 * (1 - spec_)))})
    missed[f'{target:.0%} target'] = {c: float((~send & (Y['test'][:, CLASSES.index(c)] == 1)).sum() / max(Y['test'][:, CLASSES.index(c)].sum(), 1))
                                      for c in reliable}
op_points = pd.DataFrame(rows).set_index('recall target (validation)')
missed_by_finding = pd.DataFrame(missed); missed_by_finding.index.name = 'finding'

sw = pd.DataFrame({'avoided': [100 * (1 - float((tp['test'] > t).mean())) for t in GRID], 'recall': [float((tp['test'][yt] > t).mean()) for t in GRID]})
fig, ax = plt.subplots(1, 2, figsize=(13, 3.9))
ax[0].plot(sw['avoided'], sw['recall'], '-', color='gray', label='test sweep')
for target, c in zip(THRESHOLDS_BY_TARGET, ('#4c72b0', '#dd8452', '#c44e52')):
    r_ = op_points.loc[f'{target:.0%}']
    ax[0].plot(r_['cloud calls avoided % (this test set)'], r_['test recall'], 'o', color=c, ms=9, label=f'{target:.0%} target (threshold {THRESHOLDS_BY_TARGET[target]:.3f})')
ax[0].set_xlabel('cloud calls avoided % (test set)'); ax[0].set_ylabel('test recall'); ax[0].set_title('Recall vs cloud calls avoided'); ax[0].legend(fontsize=8)
missed_by_finding.plot.bar(ax=ax[1], width=0.8); ax[1].set_ylabel('share of test positives kept at the fog'); ax[1].set_title('Which findings the fog keeps locally')
ax[1].tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.savefig(f'{FIG}/fog_operating_points.png', dpi=150); plt.show()
op_points.to_csv(f'{OUT}/results/fog_operating_points.csv'); missed_by_finding.to_csv(f'{OUT}/results/fog_missed_by_finding.csv')
print('Thresholds are fixed on validation only. "abnormal kept at fog" are abnormal X-rays the cloud model never sees; the fog does not declare them normal or safe.')
display(op_points.round(3).T); display(missed_by_finding.round(3))

In [ ]:
# ===== Cell 37b: Latency trials - one fixed workload, repeated passes, every component timed separately =====
WORKLOAD = test_df.sample(min(N_SYS, len(test_df)), random_state=0)          # the same images as Cell 32
N_TRIALS = 1 if SMOKE else 3
assert cloud_up(), 'cloud service is not running - run Cell 30 first'

def _link_ms(nbytes, latency_ms, mbps):                                       # same arithmetic as emulate_link (Cell 31), without sleeping
    return latency_ms + nbytes * 8 / (mbps * 1e6) * 1000

def profile_pass(df_):
    """Send EVERY image through the real fog node and the real cloud service once (threshold forced to -1) and record each component.
    Latency for any threshold and any link is then composed from these measurements (network terms are emulated, as in Cell 31)."""
    global THRESHOLD, emulate_link
    keep_thr, keep_link = THRESHOLD, emulate_link
    THRESHOLD, emulate_link = -1.0, (lambda nbytes, latency_ms, mbps: _link_ms(nbytes, latency_ms, mbps))
    out = []
    try:
        for _, row in df_.iterrows():
            r = fog_node(png_bytes(row['Image Index']), mode='fog+cloud', latency_ms=0, mbps=1e9, explain=True)
            tm = r['timings_ms']; assert 'cloud round trip (compute)' in tm, r['decision']
            out.append({'image': row['Image Index'], 'abnormal': int(row['abnormal']), 'p_abnormal': r['p_abnormal'],
                        'fog_preprocess_ms': tm['fog preprocess'], 'fog_inference_ms': tm['fog preprocess'] + tm['fog triage (MobileNetV3, CPU)'],
                        'cloud_inference_ms': tm['cloud round trip (compute)'], 'up_bytes': r['uplink_bytes'], 'down_bytes': r['downlink_bytes']})
    finally:
        THRESHOLD, emulate_link = keep_thr, keep_link
    return pd.DataFrame(out)

profile_pass(WORKLOAD.head(3))                                                # warm-up, discarded
trials = []
for k in range(N_TRIALS):
    t0 = time.time(); p_ = profile_pass(WORKLOAD); p_['trial'] = k; trials.append(p_)
    print(f'  trial {k + 1}/{N_TRIALS} done in {time.time() - t0:.0f} s')
profile = pd.concat(trials, ignore_index=True)

# consistency check: the fog node (CPU, one image at a time) vs the batched GPU probabilities the thresholds were chosen on
pos = {n: i for i, n in enumerate(test_df['Image Index'])}
p_ref = np.array([tp['test'][pos[n]] for n in profile['image']])
print('fog-node vs batch probability, max |difference|: %.4f' % np.abs(profile['p_abnormal'].values - p_ref).max())
for tg, th in THRESHOLDS_BY_TARGET.items():
    print(f'  {tg:.0%} target: {100 * ((profile["p_abnormal"].values > th) != (p_ref > th)).mean():.1f}% of images route differently in the live node')

LINKS = {'good link (10 ms, 50 Mbps)': (10, 50), 'average link (40 ms, 10 Mbps)': (40, 10), 'poor link (150 ms, 1 Mbps)': (150, 1)}
POLICIES = {'cloud-only (no fog)': None}
POLICIES.update({f'fog + cloud, {t:.0%} recall target': THRESHOLDS_BY_TARGET[t] for t in (0.85, 0.90, 0.95)})
COMPS = ['fog inference ms', 'network delay ms (emulated, round trip)', 'upload ms (emulated)', 'cloud inference ms', 'download ms (emulated)']
def compose(p_, thr, latency_ms, mbps):
    d = p_.copy(); send = np.ones(len(d), bool) if thr is None else (d['p_abnormal'].values > thr)
    d['sent'] = send
    d['fog inference ms'] = d['fog_preprocess_ms'] if thr is None else d['fog_inference_ms']       # cloud-only has no triage model
    d['network delay ms (emulated, round trip)'] = np.where(send, 2.0 * latency_ms, 0.0)
    d['upload ms (emulated)'] = np.where(send, d['up_bytes'] * 8 / (mbps * 1e6) * 1000, 0.0)
    d['cloud inference ms'] = np.where(send, d['cloud_inference_ms'], 0.0)
    d['download ms (emulated)'] = np.where(send, d['down_bytes'] * 8 / (mbps * 1e6) * 1000, 0.0)
    d['total ms'] = d[COMPS].sum(axis=1)
    return d

rows, brk = [], []
for link, (lat, mb) in LINKS.items():
    for pol, thr in POLICIES.items():
        per = [compose(g, thr, lat, mb) for _, g in profile.groupby('trial')]
        pooled = pd.concat(per)
        means = [d['total ms'].mean() for d in per]
        rows.append({'link': link, 'policy': pol, 'passes': len(per), 'images per pass': len(per[0]),
                     'cloud-call rate %': 100 * pooled['sent'].mean(),
                     'abnormal reaching cloud %': 100 * pooled.loc[pooled['abnormal'] == 1, 'sent'].mean(),
                     'abnormal kept at fog (per pass)': int(((pooled['abnormal'] == 1) & ~pooled['sent']).sum() / len(per)),
                     'mean latency ms': float(np.mean(means)), 'sd of mean over passes ms': float(np.std(means, ddof=1)) if len(per) > 1 else np.nan,
                     'p50 ms': pooled['total ms'].quantile(0.50), 'p95 ms': pooled['total ms'].quantile(0.95), 'p99 ms': pooled['total ms'].quantile(0.99)})
        brk.append({'link': link, 'policy': pol, **{c: pooled[c].mean() for c in COMPS}, 'total ms': pooled['total ms'].mean()})
latency_trials = pd.DataFrame(rows).set_index(['link', 'policy']); latency_breakdown = pd.DataFrame(brk).set_index(['link', 'policy'])

AVG = 'average link (40 ms, 10 Mbps)'; lat_a, mb_a = LINKS[AVG]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
dist = {pol: pd.concat([compose(g, thr, lat_a, mb_a) for _, g in profile.groupby('trial')])['total ms'].values for pol, thr in POLICIES.items()}
ax[0].boxplot(list(dist.values()), showfliers=False, showmeans=True)
ax[0].set_xticks(range(1, len(dist) + 1)); ax[0].set_xticklabels([p.replace('fog + cloud, ', 'fog+cloud\n').replace(' (no fog)', '\n(no fog)') for p in dist], fontsize=8)
ax[0].set_ylabel('response time per X-ray (ms)'); ax[0].set_title(f'Latency distribution, {AVG}')
bk = latency_breakdown.loc[AVG][COMPS]; bk.index = [p.replace('fog + cloud, ', 'fog+cloud ') for p in bk.index]
bk.plot.barh(stacked=True, ax=ax[1]); ax[1].set_xlabel('mean ms per X-ray'); ax[1].set_title('Where the time goes'); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{FIG}/system_latency_trials.png', dpi=150); plt.show()
profile.to_csv(f'{OUT}/results/system_profile.csv', index=False)
latency_trials.to_csv(f'{OUT}/results/latency_trials.csv'); latency_breakdown.to_csv(f'{OUT}/results/latency_breakdown.csv')
print(f'Workload: the same {len(WORKLOAD)} test images in every pass, {N_TRIALS} passes. Fog and cloud inference are measured on this machine; '
      'network delay, upload and download are EMULATED from payload size, latency and bandwidth (no loss, no jitter).')
display(latency_trials.round(1)); display(latency_breakdown.round(1))

In [ ]:
# ===== Cell 37c: Do the image borders carry information? Frame-only baseline and controlled perturbations =====
from IPython.display import display
B = int(round(0.08 * 256))                                                    # the same 8% frame the zone map uses (Cell 29)
FRAME = np.zeros((256, 256), bool); FRAME[:B] = True; FRAME[-B:] = True; FRAME[:, :B] = True; FRAME[:, -B:] = True
def keep_frame_only(a):                                                       # frame kept, interior replaced by the frame's mean grey level
    o = a.copy(); o[:, ~FRAME] = a[:, FRAME].mean(1, keepdims=True).astype(a.dtype); return o
def remove_frame(a):                                                          # frame replaced by the interior's mean grey level
    o = a.copy(); o[:, FRAME] = a[:, ~FRAME].mean(1, keepdims=True).astype(a.dtype); return o
def swap_frames(a, seed=0):                                                   # every image receives another test image's frame
    perm = np.random.default_rng(seed).permutation(len(a)); o = a.copy(); o[:, FRAME] = a[perm][:, FRAME]; return o
# ---- training and evaluation (needs the notebook's models) ----
def auc_ci(y, p, n=500, seed=0):
    rng = np.random.default_rng(seed); a = []
    for _ in range(50 if SMOKE else n):
        s = rng.integers(0, len(y), len(y))
        if 0 < y[s].sum() < len(s): a.append(roc_auc_score(y[s], p[s]))
    return roc_auc_score(y, p), np.percentile(a, [2.5, 97.5])

def train_binary(tag, transform):
    """Same MobileNetV3-Small triage recipe as Cell 24, but trained and tested on transformed images."""
    set_seed(42); m = TriageNet(CFG['pretrained']).to(device)
    dsx = {k: CXRDataset(transform(imgs[k]), Y[k], YA[k]) for k in ('train', 'val', 'test')}
    crit_ = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - YA['train'].mean()) / max(YA['train'].mean(), 1e-6), device=device))
    def step_(b):
        x, _, a = b; x, a = x.to(device), a.to(device); return crit_(m(x).float(), a), len(x)
    ck_ = f'{OUT}/checkpoints/triage_{tag}.pt'
    if REUSE_SAVED and os.path.exists(ck_): m.load_state_dict(torch.load(ck_, map_location=device)); print(f'  loaded saved {tag} model')
    else:
        fit(m, loader(dsx['train'], True, 64), loader(dsx['val'], bs=64), step_, 1 if SMOKE else 8, 3e-4, patience=3, name=tag, amp=AMP)
        torch.save(m.state_dict(), ck_)
    m.eval()
    with torch.no_grad(): return torch.cat([torch.sigmoid(m(x.to(device)).float()).cpu() for x, _, _ in loader(dsx['test'], bs=64)]).numpy()

p_frame = train_binary('frame_only', keep_frame_only); p_interior = train_binary('interior_only', remove_frame)
bin_rows = {}
for name, p in (('full image (fog triage, Cell 24)', tp['test']), ('interior only (frame removed)', p_interior), ('frame only (interior removed)', p_frame)):
    a, (lo, hi) = auc_ci(YA['test'], p); bin_rows[name] = {'abnormal-vs-normal AUROC (test)': a, '95% CI low': lo, '95% CI high': hi}
border_baseline = pd.DataFrame(bin_rows).T

@torch.no_grad()
def cloud_probs(arr, bs=64):
    """Cloud model (backbone + frozen PCA + the three quantum-attention heads) on a batch of 256-px images."""
    backbone.eval(); outs = []
    for s in range(0, len(arr), bs):
        x = torch.stack([to_input(a) for a in arr[s:s + bs]]).to(device)
        tok, pooled = backbone.tokens(x); tok, pooled = tok.float().cpu(), pooled.float().cpu()
        q = cloud_model.pca(pooled)
        outs.append(torch.sigmoid(torch.stack([h(tok, pooled, q) for h in cloud_model.heads]).mean(0)))
    return torch.cat(outs).numpy()

Xt = imgs['test']
conds = {'original': Xt, 'frame removed': remove_frame(Xt), 'frame swapped between images': swap_frames(Xt), 'frame only (interior removed)': keep_frame_only(Xt)}
P = {k: cloud_probs(v) for k, v in conds.items()}
rows = []
for k, p in P.items():
    d, (lo, hi) = (0.0, (0.0, 0.0)) if k == 'original' else paired_bootstrap(Y['test'], P['original'], p, reliable)
    rows.append({'condition': k, 'mean AUROC (reliable classes)': mean_auroc(Y['test'], p, reliable), 'change vs original': d,
                 '95% CI low': lo, '95% CI high': hi, 'mean |change in probability|': float(np.abs(p - P['original']).mean()),
                 'top finding changed %': 100 * float((p.argmax(1) != P['original'].argmax(1)).mean())})
border_perturbation = pd.DataFrame(rows).set_index('condition')

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
bb_ = border_baseline; ax[0].bar(range(3), bb_.iloc[:, 0], yerr=[bb_.iloc[:, 0] - bb_['95% CI low'], bb_['95% CI high'] - bb_.iloc[:, 0]], color=['#4c72b0', '#55a868', '#c44e52'])
ax[0].axhline(0.5, ls='--', c='gray'); ax[0].set_xticks(range(3)); ax[0].set_xticklabels(['full', 'interior only', 'frame only'])
ax[0].set_ylim(0.4, 1); ax[0].set_ylabel('AUROC, abnormal vs normal'); ax[0].set_title('Fog triage trained on different image parts')
bp_ = border_perturbation['mean AUROC (reliable classes)']; ax[1].bar(range(len(bp_)), bp_, color='#4c72b0'); ax[1].axhline(0.5, ls='--', c='gray')
ax[1].set_xticks(range(len(bp_))); ax[1].set_xticklabels([i.replace(' (', '\n(').replace(' between', '\nbetween') for i in bp_.index], fontsize=8)
ax[1].set_ylim(0.4, 0.8); ax[1].set_ylabel('mean AUROC (reliable classes)'); ax[1].set_title('Cloud model under frame perturbations')
plt.tight_layout(); plt.savefig(f'{FIG}/border_checks.png', dpi=150); plt.show()
border_baseline.to_csv(f'{OUT}/results/border_only_baseline.csv'); border_perturbation.to_csv(f'{OUT}/results/border_perturbation.csv')
fo = border_baseline.iloc[2]
print(f'Frame-only model: AUROC {fo.iloc[0]:.3f} (95% CI {fo["95% CI low"]:.3f}-{fo["95% CI high"]:.3f}; 0.5 = chance). ' +
      ('The border alone predicts abnormality, so border artifacts DO carry label information.' if fo['95% CI low'] > 0.5 else 'The border alone is not distinguishable from chance.'))
print('Interpretation: a gray frame is itself a distribution shift, so read "frame removed" with care; "frame swapped" keeps realistic frames and is the cleaner test of '
      'whether predictions depend on image-specific border content. Anatomical plausibility stays low (Cell 35) - keep that result in the report.')
display(border_baseline.round(3)); display(border_perturbation.round(3))

## Part J · End-to-end clinical decision reports and results

In [ ]:
# ===== Cell 38: End-to-end clinical reports through the running fog → cloud system =====
def show_report(rec, truth=None):
    print('=' * 78); print('QFOG CLINICAL DECISION-SUPPORT REPORT  (research prototype, not a diagnosis)')
    if truth: print('Ground truth (dataset label):', truth)
    if rec['p_abnormal'] is not None: print(f"Fog triage p(abnormal) = {rec['p_abnormal']:.2f}  (threshold {THRESHOLD})")
    print('Decision:', rec['decision'])
    if 'cloud' in rec:
        for c, p in rec['cloud']['findings']: print(f'   {c:<20s} {p:.2f}')
        print(); print(rec['cloud'].get('reasoning', '').replace('**', ''))
    print('Timings (ms):', {k: round(v) for k, v in rec['timings_ms'].items()})
    if 'cloud' in rec and 'heatmap_png' in rec['cloud']:
        f, a = plt.subplots(1, 2, figsize=(7, 3.5))
        for ax_, key, ttl in ((a[0], 'heatmap_png', 'Grad-CAM++ (hybrid)'), (a[1], 'quantum_attention_png', 'Quantum attention')):
            ax_.imshow(Image.open(io.BytesIO(base64.b64decode(rec['cloud'][key])))); ax_.set_title(ttl, fontsize=9); ax_.axis('off')
        plt.tight_layout(); plt.show()
for i in list(test_df.index[test_df['abnormal'] == 1][:2]) + list(test_df.index[test_df['abnormal'] == 0][:1]):
    show_report(fog_node(png_bytes(test_df.loc[i, 'Image Index'])), test_df.loc[i, 'Finding Labels'])

In [ ]:
# ===== Cell 39: Results summary (all numbers computed above) =====
att_q = comparison.loc['Attention hybrid (quantum)']; att_c = comparison.loc['Attention + classical control']
sig = significance.loc['Attention hybrid (quantum)  vs  Attention + classical control']
summary = pd.DataFrame([
    ('Dataset', f"NIH ChestX-ray14 sample: {len(df)} images, {df['Patient ID'].nunique()} patients, patient-level 70/10/20 split"),
    ('Cloud backbone mean AUROC (reliable classes, test)', f"{comparison.iloc[0]['mean AUROC (reliable)']:.3f}"),
    ('Attention hybrid (8-qubit VQC), mean ± std over seeds', f"{att_q['mean AUROC (reliable)']:.3f} ± {att_q['std over seeds']:.3f}"),
    ('Attention + classical control, mean ± std', f"{att_c['mean AUROC (reliable)']:.3f} ± {att_c['std over seeds']:.3f}"),
    ('Quantum vs classical control (bootstrap 95% CI)', f"{sig['AUROC difference']:+.3f} [{sig['95% CI low']:+.3f}, {sig['95% CI high']:+.3f}] — "
                                                       + ('significant' if sig['statistically significant'] else 'not significant')),
    ('Best model (seed ensemble), mean AUROC', f"{comparison['mean AUROC (reliable)'].max():.3f} ({comparison['mean AUROC (reliable)'].idxmax()})"),
    ('Why 8 qubits', f"16 qubits: {qubit_study['train step time, batch 16 (s)'].iloc[-1] / qubit_study.loc[8, 'train step time, batch 16 (s)']:.0f}x slower step, "
                     f"output spread {qubit_study.iloc[-1, 1]:.3f} vs {qubit_study.loc[8].iloc[1]:.3f}" if 8 in qubit_study.index and len(qubit_study) > 1 else 'see qubit study'),
    ('Fog triage (test) recall / specificity', f"{fog_test['recall (abnormal sent to cloud)']:.1%} / {fog_test['specificity (normals kept at fog)']:.1%} at threshold {THRESHOLD}"),
    ('Fog CPU speed-up vs cloud backbone', f'{SPEEDUP:.1f}x'),
    ('Live system: cloud calls / upload / cloud compute saved', f"{SAVINGS['cloud calls avoided %']:.1f}% / {SAVINGS['upload saved %']:.1f}% / {SAVINGS['cloud compute saved %']:.1f}%"),
    ('Projected cloud calls avoided at 20% abnormal prevalence', f"{prev.loc['20%', 'cloud calls avoided %']:.1f}%"),
    ('Cloud outage', f"{int(system.loc['fog+cloud | cloud outage', 'fallbacks handled'])} of {int(system.loc['fog+cloud | cloud outage', 'images'])} flagged cases handled by fog fallback"),
    ('XAI: explanations anatomically plausible', f"{xai_summary['anatomically plausible %']:.0f}% of analysed test cases"),
    ('XAI: faithfulness (prob. drop highlighted vs random)', f"{xai_summary['prob. drop, highlighted pixels removed']:.3f} vs {xai_summary['prob. drop, random pixels removed']:.3f}"),
], columns=['Result', 'Value']).set_index('Result')
summary

In [ ]:
# ===== Cell 39a: Add the evaluation-hardening results to the summary table =====
n_abn = int(yt.sum()); extra = []
for tg in ('85%', '90%', '95%'):
    r = op_points.loc[tg]
    extra.append((f'Fog @ {tg} recall target (val-selected): test recall [95% CI] / specificity / cloud calls avoided / abnormal kept at fog',
                  f"{r['test recall']:.1%} {r['test recall 95% CI']} / {r['test specificity']:.1%} / {r['cloud calls avoided % (this test set)']:.1f}% / {int(r['abnormal kept at fog (n)'])} of {n_abn}"))
lt = latency_trials.loc[AVG]
extra.append((f'Latency, {AVG}, mean / p95 ms over {N_TRIALS} passes (network emulated)',
              '; '.join(f"{pol.replace('fog + cloud, ', 'fog+cloud ')}: {lt.loc[pol, 'mean latency ms']:.0f} / {lt.loc[pol, 'p95 ms']:.0f}" for pol in lt.index)))
extra.append(('Border check: abnormal AUROC full / interior only / frame only',
              ' / '.join(f"{v:.3f}" for v in border_baseline.iloc[:, 0])))
sw_ = border_perturbation.loc['frame swapped between images']
extra.append(('Cloud model, frame swapped between images: mean AUROC change [95% CI]', f"{sw_['change vs original']:+.3f} [{sw_['95% CI low']:+.3f}, {sw_['95% CI high']:+.3f}]"))
extra = pd.DataFrame(extra, columns=['Result', 'Value']).set_index('Result')
summary = pd.concat([summary[~summary.index.isin(extra.index)], extra])
summary

In [ ]:
# ===== Cell 40: Save every result and artefact to Drive =====
comparison.to_csv(f'{OUT}/results/model_comparison.csv'); significance.to_csv(f'{OUT}/results/significance_tests.csv')
per_class.to_csv(f'{OUT}/results/per_class_auroc.csv'); qubit_study.to_csv(f'{OUT}/results/qubit_study.csv')
head_to_head.to_csv(f'{OUT}/results/qubit_head_to_head.csv'); system.to_csv(f'{OUT}/results/system_experiment.csv')
prev.to_csv(f'{OUT}/results/prevalence_projection.csv'); xai_df.to_csv(f'{OUT}/results/xai_per_case.csv', index=False)
summary.to_csv(f'{OUT}/results/summary.csv'); split_table.to_csv(f'{OUT}/results/split_table.csv'); audit.to_frame('value').to_csv(f'{OUT}/results/dataset_audit.csv')
json.dump({'config': CFG, 'threshold': THRESHOLD, 'thresholds_by_target': {str(k): float(v) for k, v in THRESHOLDS_BY_TARGET.items()}, 'fog_test': fog_test, 'speedup': SPEEDUP, 'savings': SAVINGS,
           'xai_summary': xai_summary.to_dict(), 'reliable_classes': reliable}, open(f'{OUT}/results/run_metadata.json', 'w'), indent=2, default=float)
print('Saved results:', sorted(os.listdir(f'{OUT}/results'))); print('Saved figures:', sorted(os.listdir(FIG)))

In [ ]:
# ===== Cell 41: Export the deployment bundle for the Streamlit app =====
# Everything the stand-alone website (QFog_App) needs: trained models, configuration, result tables, figures, example X-rays.
DEP = f'{OUT}/qfog_deploy'
for d in ('models', 'results', 'figures', 'examples'): os.makedirs(f'{DEP}/{d}', exist_ok=True)
torch.save(cloud_model.state_dict(), f'{DEP}/models/cloud_model.pt')
torch.save(triage_cpu.state_dict(), f'{DEP}/models/triage.pt')
for f_ in glob.glob(f'{OUT}/results/*'): shutil.copy(f_, f'{DEP}/results/')
for f_ in glob.glob(f'{FIG}/*.png'): shutil.copy(f_, f'{DEP}/figures/')
for lab, n in ((1, 4), (0, 2)):
    for i in test_df.index[test_df['abnormal'] == lab][:n]:
        nm = test_df.loc[i, 'Image Index']
        shutil.copy(img_path[nm], f"{DEP}/examples/{'abnormal' if lab else 'normal'}__{test_df.loc[i, 'Finding Labels'].replace('|', '+')}__{nm}")
deploy_cfg = {'classes': CLASSES, 'backbone': CFG['backbone'], 'n_qubits': NQ, 'n_layers': NL, 'n_heads': len(CFG['seeds']),
              'feature_dim': DIM, 'threshold': THRESHOLD, 'thresholds_by_target': {str(k): float(v) for k, v in THRESHOLDS_BY_TARGET.items()}, 'fog_test': fog_test, 'speedup': float(SPEEDUP), 'savings': SAVINGS,
              'xai_summary': xai_summary.to_dict(), 'reliable_classes': reliable,
              'dataset': {'images': int(len(df)), 'patients': int(df['Patient ID'].nunique()), 'classes': len(CLASSES)}}
json.dump(deploy_cfg, open(f'{DEP}/deploy_config.json', 'w'), indent=2, default=float)
shutil.make_archive(f'{PROJECT_DIR}/qfog_deploy', 'zip', DEP)
print('Deployment bundle written to', f'{PROJECT_DIR}/qfog_deploy.zip',
      f'({os.path.getsize(f"{PROJECT_DIR}/qfog_deploy.zip") / 1e6:.0f} MB) - download it and unzip into QFog_App/deploy/')

## Part K · Live demo dashboard
Run the next cell and open the **public link** (`https://xxxx.gradio.live`). Tabs:
**Dashboard · Dataset explorer · EDA · Quantum design (16→8 qubits) · Model performance · Fog–cloud system · Explainable AI · Live X-ray diagnosis**.
In *Live X-ray diagnosis*, upload any frontal chest X-ray (PNG/JPG) or click an example: the image goes through the real fog node and the real
cloud service, and the report, heatmaps, reasoning and timings appear. Sliders change the network; a checkbox simulates a cloud outage.
The link stays live while this Colab tab is open.

In [ ]:
# ===== Cell 42: Launch the dashboard (public link) =====
import gradio as gr
examples = []
for lab, n in ((1, 4), (0, 2)):
    for i in test_df.index[test_df['abnormal'] == lab][:n]:
        dst = f"{OUT}/demo_examples/{test_df.loc[i, 'Image Index']}"
        if not os.path.exists(dst): shutil.copy(img_path[test_df.loc[i, 'Image Index']], dst)
        examples.append([dst])
def fig_path(n): p = f'{FIG}/{n}'; return p if os.path.exists(p) else None
def b64_img(s): return np.array(Image.open(io.BytesIO(base64.b64decode(s))))

def live(pil, mode, latency, bandwidth, outage):
    if pil is None: return 'Upload a chest X-ray or pick an example.', None, None, None
    buf = io.BytesIO(); pil.convert('L').save(buf, format='PNG')
    rec = fog_node(buf.getvalue(), mode=mode, latency_ms=latency, mbps=bandwidth, cloud_down=outage)
    md_ = '### 1 · Fog node\n'
    md_ += (f"p(abnormal) = **{rec['p_abnormal']:.2f}** (threshold {THRESHOLD})\n\n" if rec['p_abnormal'] is not None else 'Cloud-only mode: no fog screening.\n\n')
    md_ += f"**Decision:** {rec['decision']}\n\n"
    heat = att = table = None
    if 'cloud' in rec:
        cl = rec['cloud']
        md_ += '### 2 · Cloud: quantum–classical diagnosis\n' + '\n'.join(f'- **{c}**: {p:.2f}' for c, p in cl['findings'])
        md_ += '\n\n### 3 · Explainable-AI reasoning\n' + cl.get('reasoning', '')
        heat, att = b64_img(cl['heatmap_png']), b64_img(cl['quantum_attention_png'])
        table = pd.DataFrame(sorted(cl['probs'].items(), key=lambda kv: -kv[1]), columns=['finding', 'probability']).round(3)
    md_ += '\n\n### Timing\n' + '\n'.join(f'- {k}: {v:.0f} ms' for k, v in rec['timings_ms'].items())
    md_ += f"\n\nData sent to cloud: {rec.get('uplink_bytes', 0) / 1000:.1f} KB (original image {rec['device→fog bytes'] / 1000:.0f} KB)"
    return md_, heat, att, table

with gr.Blocks(title='QFog — Quantum-Enhanced Fog Learning') as demo:
    gr.Markdown('# Quantum-Enhanced Fog Learning for Early Pulmonary Disorder Detection\n'
                'Fog triage (MobileNetV3, CPU) → cloud DenseNet121 + 8-qubit PennyLane VQC + quantum attention → Grad-CAM++ & reasoning. '
                '*Academic research prototype — not a medical device.*')
    with gr.Tab('Dashboard'):
        with gr.Row():
            for label, val in [('X-rays', len(df)), ('Patients', df['Patient ID'].nunique()), ('Findings', 14), ('Qubits', NQ),
                               ('Fog recall', f"{fog_test['recall (abnormal sent to cloud)']:.0%}"), ('Cloud calls avoided', f"{SAVINGS['cloud calls avoided %']:.0f}%")]:
                gr.Markdown(f'<div style="border:1px solid #ccc;border-radius:10px;padding:10px;text-align:center"><b>{label}</b><h2>{val}</h2></div>')
        gr.Markdown('**Workflow:** X-ray → fog preprocessing → fog triage → (if abnormal) network → cloud backbone → PCA → 8-qubit VQC → quantum attention → 14 findings → Grad-CAM++ + reasoning → report')
        gr.Dataframe(summary.reset_index(), label='Key results (computed in this notebook)')
    with gr.Tab('Dataset explorer'):
        gr.Dataframe(audit.to_frame('value').reset_index(), label='Dataset audit'); gr.Dataframe(split_table.reset_index(), label='Patient-level split')
        gr.Image(fig_path('eda_examples.png'), label='One example per class')
    with gr.Tab('EDA'):
        gr.Image(fig_path('eda_class_distribution.png')); gr.Image(fig_path('eda_cooccurrence_demographics.png')); gr.Image(fig_path('augmentation.png'))
    with gr.Tab('Quantum design (16 → 8 qubits)'):
        gr.Markdown('**Why 8 qubits?** More qubits keep slightly more PCA information, but outputs bunch together across patients, gradients vanish '
                    '(barren plateau) and simulation cost doubles per qubit. 8 is the largest circuit that stays trainable and affordable.')
        gr.Image(fig_path('qubit_study.png')); gr.Dataframe(qubit_study.reset_index().round(4)); gr.Dataframe(head_to_head.reset_index(), label='8 vs 16 qubits, same training budget')
    with gr.Tab('Model performance'):
        gr.Dataframe(comparison.reset_index().round(4), label='Test mean AUROC'); gr.Dataframe(significance.reset_index(), label='Bootstrap significance tests')
        gr.Image(fig_path('per_class_auroc.png')); gr.Image(fig_path('baseline_roc.png'))
    with gr.Tab('Fog–cloud system'):
        gr.Image(fig_path('fog_triage.png')); gr.Dataframe(system.reset_index(), label='Running system'); gr.Image(fig_path('system_results.png'))
        gr.Dataframe(prev.reset_index(), label='Projected savings at screening prevalence'); gr.Dataframe(speed.reset_index(), label='Speed')
    with gr.Tab('Explainable AI'):
        gr.Image(fig_path('xai_gradcam_quantum_attention.png')); gr.Dataframe(xai_summary.to_frame('value').reset_index(), label='XAI over test cases')
        gr.Image(fig_path('xai_global.png')); gr.Image(fig_path('xai_qubit_attribution.png')); gr.Dataframe(shortcut.reset_index(), label='Shortcut check')
    with gr.Tab('Live X-ray diagnosis'):
        with gr.Row():
            with gr.Column():
                inp = gr.Image(type='pil', label='Upload a chest X-ray')
                mode = gr.Radio(['fog+cloud', 'cloud-only'], value='fog+cloud', label='System mode')
                lat = gr.Slider(5, 300, value=40, step=5, label='Network latency (ms)'); bw = gr.Slider(0.5, 100, value=10, label='Bandwidth (Mbps)')
                outage = gr.Checkbox(label='Simulate cloud outage'); btn = gr.Button('Run diagnosis', variant='primary')
                gr.Examples(examples, inputs=inp)
            with gr.Column():
                rep = gr.Markdown()
                with gr.Row(): heat = gr.Image(label='Grad-CAM++ (through the quantum hybrid)'); att = gr.Image(label='Quantum attention map')
                tab = gr.Dataframe(label='All 14 findings')
        btn.click(live, [inp, mode, lat, bw, outage], [rep, heat, att, tab])

print(live(Image.open(examples[0][0]), 'fog+cloud', 40, 10, False)[0][:400])        # self-test before launching
if not SMOKE:
    demo.launch(share=True, debug=False)

## Part L · Conclusion, limitations and viva questions
**Conclusion.** The project delivers a complete, running two-tier system: a CPU fog node screens every X-ray and keeps *low-risk* cases local (a screening result, never a clearance); flagged
cases cross an (emulated) network to a cloud service where a DenseNet121 backbone, an 8-qubit PennyLane circuit and quantum-attention fusion
predict 14 findings; every cloud diagnosis carries Grad-CAM++ heatmaps computed through the quantum circuit, a quantum-attention map and a
plain-language reasoning report with plausibility, shortcut and faithfulness checks. All claims are measured on a patient-level held-out test set,
against a matched classical control, with seeds and bootstrap confidence intervals. **The quantum layer is an experimental component: the classical model is the stronger
predictive baseline in these results, and the quantum hypothesis is not supported.** Report the quantum result exactly as Cell 22 prints it.

**Central limitation: the fog can miss abnormal X-rays.** The fog screen is a recall-limited filter. At every operating point in Cell 37a, some abnormal images are kept at the fog
and never reach the cloud model, and the cloud-outage fallback does not change that. Raising the recall target (85% → 90% → 95%) reduces the misses and also the cloud-call savings; Cell 37a reports both.
The fog therefore never labels an image "normal": it reports *low-risk on the fog screen, not reviewed by the cloud model*.

**Other limitations.** Small NIH sample (5,606 images) with NLP-mined, noisy labels; quantum circuits are simulated (no hardware noise);
the network is emulated (latency and bandwidth only: no packet loss or jitter) and fog and cloud run on one machine; the zone-based reasoning is a heuristic plausibility check, not a radiologist,
and only a small share of explanations were anatomically plausible (Cell 35); border artifacts may carry label information (Cell 37c);
the sample is ~50% abnormal, so the fog saving is reported both as measured and projected to screening prevalence.

**Viva questions — short answers**
1. **Why fog computing?** Most X-rays are low-risk; a cheap CPU model at the edge avoids sending them to the cloud, saving bandwidth, compute and latency, and keeps working during cloud outages. It is a screening filter, not a diagnosis.
2. **Why a recall-biased threshold?** Missing an abnormal case is worse than an extra cloud call; the threshold is fixed on validation, never on test. Cell 37a shows the price of each recall target.
3. **Why patient-level splits?** The same patient's images in train and test would let the model recognise the patient, not the disease.
4. **Why PCA before the circuit?** 1,024 features cannot be encoded on 8 qubits; PCA keeps the directions of largest variance.
5. **What is data re-uploading?** Encoding the input again in every layer, which increases what a small circuit can represent.
6. **Why the CNOT ring?** It entangles neighbouring qubits so the circuit can model interactions between features.
7. **Why 8 qubits, not 16?** Measured in Cells 17–18: simulation cost grows steeply with qubits, and output spread and gradient variance shrink (read the exact numbers from your table); 8 was the best accuracy-per-cost point we tested, not a proven optimum.
8. **What is a barren plateau?** In large random circuits gradients shrink exponentially with qubits, so training stalls.
9. **Why a classical control?** It is identical except the VQC; only this comparison isolates the quantum circuit's contribution.
10. **What does the attention fusion do?** The quantum features form a query that weights 49 image regions; the weighted region features join the classifier.
11. **What is Grad-CAM++?** It weights the last feature maps by gradients of a class score to show which image regions raised that score; here gradients pass through the quantum circuit.
12. **How do you know the explanation is faithful?** Hiding the highlighted pixels lowers the probability more than hiding random pixels (Cell 35). Faithful is not the same as anatomically plausible: plausibility was low.
13. **What is shortcut learning?** The model using non-medical cues (text markers, borders). Cell 37c tests it directly with a frame-only model and frame perturbations.
14. **Is this a diagnostic tool?** No — it is a research prototype on a public benchmark without clinical validation.
15. **Can the fog miss a sick patient?** Yes. At the validation-selected threshold some abnormal images are kept at the fog and never seen by the cloud model; Cell 37a counts them for each recall target.
16. **Did the quantum model help?** No. It scored below the matched classical control and the classical baseline; we report it as an experimental component with no demonstrated accuracy benefit here.
17. **What is measured and what is emulated?** Fog and cloud inference are measured (on one machine); network delay, upload and download are computed from payload size, latency and bandwidth.
